# Klasifikasi Multi-Label Insiden Siber di Bawah Pergeseran Distribusi Temporal
### Studi berbasis reproduktifitas pada EuRepoC Global Dataset

**Scope:** Kami mengklasifikasikan insiden siber berdasarkan `incident_type` — target
**multi-label** (satu insiden bisa sekaligus mis. *Disruption* **dan** *Hijacking with
Misuse*) — dan mempelajari apakah model yang dilatih pada insiden lama masih bekerja pada
insiden yang lebih baru, dan seberapa besar performanya menurun seiring waktu ("pergeseran
distribusi temporal").

**Aturan dasar yang dipegang teguh di seluruh notebook:**
- Setiap angka di bawah ini berasal dari kode yang dijalankan di notebook ini, pada CSV asli
  — tidak ada yang diketik manual atau dikarang.
- Semua preprocessing (encoder, threshold, feature selection) di-*fit* hanya pada partisi
  **training**.
- **Data 2024 tidak pernah dipakai untuk membuat keputusan apa pun** -- tidak untuk
  memilih model, memilih fitur/vocabulary (termasuk `TOP_K_REGIONS` dan target
  `MultiLabelBinarizer`, keduanya di-fit hanya di TRAIN), menyetel threshold, atau
  memilih strategi/fraksi adaptasi (dipilih dari sisa data 2023 saja). 2024 memang
  muncul sebagai titik pembanding DESKRIPTIF di analisis distribution-shift dan
  robustness-drop (Bagian 18-19), karena eval brief secara eksplisit mewajibkan
  perbandingan itu -- tapi tidak ada satu pun angka dari perbandingan itu yang
  dipakai untuk mengubah keputusan apa pun. Walk-forward validation (Bagian 17),
  explainability & error-analysis utama (Bagian 22-23), dan uji statistik (Bagian
  24) dibatasi tegas sampai tahun 2023; panel 2024 untuk explainability/error
  analysis dipindah ke Bagian 26 (Final Results), dihitung ulang di sana SETELAH
  semua keputusan (model, fitur, strategi adaptasi) benar-benar dibekukan.
- Jika hasil verifikasi kami terhadap klaim di audit brief berbeda dari angka yang diberikan,
  keduanya ditampilkan dan perbedaannya dijelaskan, bukan diam-diam memilih salah satu.

**Envirollment Notes:** Notebook ini dirancang untuk dijalankan dengan `pip install -r
requirements.txt`, yang mencakup `xgboost`, `lightgbm`, dan `shap` sehingga perbandingan
model dan explainability lengkap. Jika salah satu paket itu tidak terpasang, sel-sel terkait
(Bagian 16 dan Bagian 22) otomatis jatuh kembali (fallback) ke fungsionalitas yang lebih
terbatas (model roster lebih pendek / permutation importance) dan mencetak peringatan yang
jelas — notebook tidak akan gagal total, tapi perbandingannya tidak akan selengkap yang
seharusnya. Notebook ini terakhir dieksekusi penuh di lingkungan tanpa akses internet
(sehingga tiga paket di atas belum terinstal saat itu) — setelah paket-paket tersebut
dipasang, jalankan ulang semua sel dari atas ke bawah agar seluruh output mencerminkan
model dan hasil explainability yang lengkap.


In [ ]:
import warnings, json, pickle, sys, ast
import os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance, spearmanr, wilcoxon
import sklearn, scipy
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, hamming_loss, precision_score, recall_score,
                              balanced_accuracy_score, confusion_matrix, make_scorer,
                              brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# S6.2 -- output dir dapat di-override; tidak boleh gagal bila path default tidak ada.
WORK_DIR = os.environ.get('EUREPOC_WORK_DIR', os.path.join(os.getcwd(), 'out'))
os.makedirs(WORK_DIR, exist_ok=True)

print('--- Reproducibility / environment info ---')
print('python      :', sys.version.split()[0])
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('scipy       :', scipy.__version__)
print('matplotlib  :', matplotlib.__version__)
print('seaborn     :', sns.__version__)
print('random seed used everywhere below:', RANDOM_SEED)

try:
    import xgboost
    from xgboost import XGBClassifier
    HAS_XGB = True
    print('xgboost     :', xgboost.__version__)
except Exception as e:
    HAS_XGB = False
    print('xgboost     : NOT AVAILABLE ->', e)
try:
    import lightgbm
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
    print('lightgbm    :', lightgbm.__version__)
except Exception as e:
    HAS_LGBM = False
    print('lightgbm    : NOT AVAILABLE ->', e)
try:
    import shap
    HAS_SHAP = True
    print('shap        :', shap.__version__)
except Exception as e:
    HAS_SHAP = False
    print('shap        : NOT AVAILABLE ->', e)

print()
print('This notebook was authored with pip install -r requirements.txt, which pins xgboost, '
      'lightgbm and shap. If any of the three above show NOT AVAILABLE, install requirements.txt '
      'first -- Section 16 (model comparison) and Section 22 (explainability) both fall back to '
      'reduced functionality automatically, but the comparison will not be complete without them.')


## 2. Pertanyaan Riset dan Hipotesis

**RQ1 (Tugas).** Bisakah `incident_type` diprediksi dari informasi yang secara masuk akal
tersedia pada atau segera setelah observasi awal insiden?

**RQ2 (Pergeseran).** Seberapa besar distribusi gabungan target dan fitur early-available
berubah dari periode training (<=2021) hingga 2022, 2023, dan 2024?

**RQ3 (Degradasi).** Apakah performa pada tahun-tahun berikutnya menurun relatif terhadap
referensi tahun validasi, dan apakah besarnya penurunan itu sejalan dengan pergeseran
distribusi yang terukur?

**RQ4 (Adaptasi).** Jika sepotong kecil data 2023 yang tersusun kronologis disediakan untuk
adaptasi, seberapa besar degradasi bisa dipulihkan, dan apakah *cara* potongan itu digunakan
(retraining biasa vs. recency-weighting) berpengaruh?

**RQ5 (Stabilitas).** Apakah fitur-fitur yang mendorong prediksi stabil sepanjang waktu, dan
apakah kesalahan model berhubungan secara sistematis dengan label, tahun, kategori target,
wilayah, atau pola missingness tertentu?

**Hipotesis kerja** (untuk dikonfirmasi atau ditolak di bawah, bukan diasumsikan benar):
- H1: Performa pada 2023/2024 lebih rendah dibanding tahun validasi 2022.
- H2: Besarnya penurunan itu berasosiasi dengan besarnya pergeseran distribusi yang terukur,
  meski tidak harus monotonik mengingat ukuran sampel yang sedang.
- H3: Sejumlah kecil data adaptasi kronologis terbaru memulihkan performa yang hilang secara
  parsial, bukan sepenuhnya.
- H4: Peringkat feature importance secara umum, meski tidak sempurna, tetap stabil antar
  periode.


## 3. Memuat Data

Memuat CSV mentah tanpa transformasi apa pun dulu, agar Bagian 4 (kualitas data) mengaudit
data dalam bentuk aslinya.


In [ ]:
# S6.2 -- lokasi data & output tidak hardcode ke satu mesin.
# WORK_DIR tetap bisa di-override lewat env var; jika path lama tidak ada,
# fallback ke dataset lokal notebook agar notebook dapat dijalankan ulang di mesin lain.
# WORK_DIR sudah ditetapkan di Bagian 1.

_CANDIDATES = [
    os.path.join(WORK_DIR, 'eurepoc_global_dataset_1_3.csv'),
    os.path.join(os.getcwd(), 'dataset', 'eurepoc_global_dataset_1_3.csv'),
    os.path.join(os.path.dirname(os.path.abspath('.')), 'notebook', 'dataset',
                 'eurepoc_global_dataset_1_3.csv'),
]
RAW_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if RAW_PATH is None:
    raise FileNotFoundError('eurepoc_global_dataset_1_3.csv tidak ditemukan. '
                            f'Dicari di: {_CANDIDATES}')
print('RAW_PATH:', RAW_PATH)
# keep_default_na=False + na_values=[''] : pandas secara default memperlakukan
# string 'NA' sebagai NaN, padahal 'NA' adalah kode ISO Namibia. Konversi itu
# terjadi SEBELUM kode kita sempat berjalan, sehingga invisibel di sel manapun.
df_raw = pd.read_csv(RAW_PATH, low_memory=False, keep_default_na=False, na_values=[''])
print('Raw shape:', df_raw.shape)
print('Columns (%d):' % len(df_raw.columns))
print(list(df_raw.columns))


## 4. Kualitas Data

Memeriksa klaim-klaim di audit brief terhadap file yang sebenarnya.


In [ ]:
# ============================================================
# DATA QUALITY AUDIT
# ============================================================

# Expected audit claims
audit_claims = {
    'shape (rows, cols)': (3414, 85),
    'duplicate rows': 0,
    'duplicate incident_id': 0,
    'casualties fully missing': True,
}

# Missing-value tokens used for audit
MISSING_TOKENS = {
    'not available',
    'unknown',
    'n/a',
    'na',
    '',
    'nan',
    'none'
}

# Verify required columns
required_columns = ['incident_id', 'casualties']

missing_columns = [
    col for col in required_columns
    if col not in df_raw.columns
]

if missing_columns:
    raise KeyError(f"Required columns not found: {missing_columns}")

# Actual dataset conditions
actual = {
    'shape (rows, cols)': df_raw.shape,
    'duplicate rows': int(df_raw.duplicated().sum()),
    'duplicate incident_id': int(df_raw['incident_id'].duplicated().sum()),
    'casualties fully missing': bool(
        (
            df_raw['casualties']
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(MISSING_TOKENS)
        ).all()
    )
}

# Compare claims with actual data
check_df = pd.DataFrame({
    'audit_claim': audit_claims,
    'verified_actual': actual
})

check_df['match'] = (
    check_df['audit_claim'].astype(str)
    == check_df['verified_actual'].astype(str)
)

print("=== DATA QUALITY AUDIT ===")
print(check_df.to_string())

# Overall audit status
if check_df['match'].all():
    print("\nAUDIT STATUS: PASS")
    print("All audit claims match the raw dataset.")
else:
    print("\nAUDIT STATUS: WARNING")
    print("Some audit claims do not match the raw dataset.")
    print("\nUnmatched items:")
    print(check_df[~check_df['match']].to_string())

**Hasil verifikasi:** shape (3414x85), nol baris duplikat, nol `incident_id` duplikat,
dan `casualties` yang 100% hilang, semuanya cocok persis. `casualties` tetap dibuang
kemudian — kolom ini juga ada di daftar leakage/exclusion.


## 5. Missingness

Tiga jenis "tidak ada nilai" di EuRepoC **tidak setara secara mekanisme**, dan draf
sebelumnya meleburnya menjadi satu `NaN`:

| token | arti | contoh jumlah |
|---|---|---|
| `Not available` | nilai tidak dikodekan/diungkapkan untuk baris tsb | 1.194 di `offline_conflict_intensity` |
| `Unknown` | koder tidak dapat menentukan nilainya | 1.373 di kolom yang sama |
| `Not attributed` | tidak ada atribusi (bukan missing teknis) | 935 di `initiator_category` |
| `#N/A` | error marker dari proses ekspor | 192 baris di `receiver_country_alpha_2_code` |

Menggabungkannya berarti model tidak pernah bisa membedakan "tidak dilaporkan" dari
"dilaporkan tapi tidak jelas" — padahal perbedaan itulah yang paling bergeser antar
periode. Karena itu setiap token dipetakan ke **sentinel eksplisit** yang dapat menjadi
level kategorinya sendiri.

Dua hal yang sengaja **tidak** digabung ke daftar missing: `Not attributed` (kategori sah
secara semantik untuk fitur atribusi) dan kode ISO `NA` (Namibia) — token `'na'` pada
daftar draf lama memakan satu baris sah (`incident_id` 4149).

Bagian ini juga menghitung missingness **per periode**, karena pola hilangnya bergerak
tajam menurut waktu — dan itulah yang membuat sebagian penurunan performa bersifat
mekanis, bukan substantif.


### S1-1 — Standardisasi token missing dan pembentukan dataset kerja tunggal

Sebelumnya notebook punya **dua** cabang data: `df` (dibuat di sini) dan `clean`
(dibuat ulang dari `df_raw` di Bagian 6). Seluruh analisis missingness berjalan di
`df`, yang tidak punya kolom `year`, dan `df` tidak pernah dipakai lagi setelah
Bagian 5 — sehingga semua statistik missingness dihitung tanpa dimensi waktu dan
masih memuat 92 baris tanpa tanggal yang dibuang di Bagian 13.

Diperbaiki: satu dataset kerja (`clean`) dibentuk **sekali** di sini, sudah memuat
`start_date_parsed`, `year`, dan `period`. Semua bagian berikutnya memakai `clean`.
Token missing tetap dinormalisasi ke sentinel eksplisit (bukan `np.nan`) agar
mekanisme missingness tetap dapat dibedakan oleh model.

In [ ]:
# ============================================================
# S1.1 -- DATASET KERJA TUNGGAL
# ============================================================

# Token missing dipetakan ke SENTINEL EKSPLISIT, bukan np.nan tunggal, supaya
# "tidak dilaporkan" dan "tidak diketahui" tidak melebur jadi satu kategori.
MISSING_SENTINELS = {
    'not available':  'MISSING_NOT_REPORTED',
    'unknown':        'MISSING_UNKNOWN',
    'not attributed': 'MISSING_NOT_ATTRIBUTED',
    '#n/a':           'MISSING_NOT_REPORTED',
    'n/a':            'MISSING_NOT_REPORTED',
    'none':           'MISSING_NOT_REPORTED',
    '':               'MISSING_NOT_REPORTED',
}
# CATATAN: 'na' SENGAJA tidak ada di peta ini. 'NA' adalah kode ISO-3166 untuk
# Namibia dan muncul sebagai nilai sah di receiver_country_alpha_2_code.

def is_text_dtype(s):
    """pandas 3 memakai dtype 'str' (bukan 'object'); dukung keduanya."""
    return str(s.dtype) in ('object', 'str', 'string') or str(s.dtype).startswith('string')

clean = df_raw.copy()

# 1) normalisasi token missing -- PER TOKEN, bukan per sel.
#    Sel multi-nilai seperti "Not available;Not available" tidak akan cocok dengan
#    pencocokan nilai penuh, sehingga tokennya lolos sebagai kategori sah. Karena itu
#    setiap sel dipecah pada ';' dan setiap token dipetakan sendiri-sendiri.
_token_counts = {}
def normalize_missing_tokens(x):
    if pd.isna(x):
        return x
    text = str(x)
    if ';' not in text:
        mapped = MISSING_SENTINELS.get(text.strip().lower())
        return mapped if mapped else text
    parts = []
    for t in text.split(';'):
        t = t.strip()
        if not t:
            continue
        parts.append(MISSING_SENTINELS.get(t.lower(), t))
    return ';'.join(parts) if parts else 'MISSING_NOT_REPORTED'

for col in clean.columns:
    if not is_text_dtype(clean[col]):
        continue
    before = clean[col]
    after = before.apply(normalize_missing_tokens)
    n_changed = int((before.astype(str) != after.astype(str)).sum())
    if n_changed:
        _token_counts[col] = n_changed
        clean[col] = after

print('Sel dengan token missing yang dinormalisasi (10 teratas):')
print(pd.Series(_token_counts).sort_values(ascending=False).head(10).to_string())

# 2) tanggal + periode -- dipindah KE ATAS supaya semua analisis berikutnya
#    sudah bisa memakai dimensi waktu.
clean['start_date_parsed'] = pd.to_datetime(clean['start_date'], format='%d.%m.%Y', errors='coerce')
clean['end_date_parsed'] = pd.to_datetime(clean['end_date'], format='%d.%m.%Y', errors='coerce')
clean['year'] = clean['start_date_parsed'].dt.year

# presisi tanggal: 01.01.YYYY adalah sentinel presisi-tahun, bukan tanggal nyata
clean['date_precision'] = np.where(
    clean['start_date_parsed'].isna(), 'unparsed',
    np.where(clean['start_date_parsed'].dt.strftime('%m%d') == '0101', 'year_only', 'day')
)

def period_of(y):
    if pd.isna(y):
        return 'UNDATED'
    if y <= 2021:
        return 'TRAIN(<=2021)'
    return str(int(y))

clean['period'] = clean['year'].apply(period_of)

PERIODS = ['TRAIN(<=2021)', '2022', '2023', '2024']

assert 'year' in clean.columns
print('\nShape:', clean.shape)
print('Distribusi period:')
print(clean['period'].value_counts().reindex(PERIODS + ['UNDATED', '2000', '2001']).dropna().to_string())
print('\nDistribusi date_precision:')
print(pd.crosstab(clean['period'], clean['date_precision']).to_string())
# Vocabulary label target (dipakai konsisten di seluruh notebook)
LABELS = ['Disruption', 'Data theft', 'Hijacking with Misuse',
          'Hijacking without Misuse', 'Ransomware', 'Data theft & Doxing']
LABEL_VOCAB = set(LABELS)
print('Label vocabulary:', len(LABELS), 'label ->', LABELS)


### S1.2 — Missingness per periode

Missingness di EuRepoC bergerak tajam menurut waktu: `cyber_conflict_issue` kosong
33% pada TRAIN tapi 83% pada 2024. Tanpa tabel ini, seluruh diskusi "model belajar
dari pola missingness" (Bagian 5) tidak punya dasar terukur, dan penurunan performa
tidak bisa dipisahkan dari hilangnya fitur.

In [ ]:
# Semua kolom kandidat dihitung, termasuk yang tidak punya NaN sama sekali,
# supaya matriks per periode lengkap dan heatmap di sel berikutnya tidak zero-size.
def is_missing_cell(x):
    """Missing = NaN, atau tidak ada satu pun token yang bukan sentinel MISSING_*."""
    if pd.isna(x):
        return True
    toks = [t.strip() for t in str(x).split(';') if t.strip()]
    return not any(not t.startswith('MISSING_') for t in toks)

MISSING_COLS = [c for c in ['receiver_category', 'receiver_subcategory', 'source_disclosure',
                            'zero_days', 'cyber_conflict_issue', 'offline_conflict_issue',
                            'offline_conflict_intensity', 'initiator_category',
                            'mitre_initial_access', 'incident_type', 'start_date', 'end_date']
                if c in clean.columns]
MISSING_COLS += [c for c in clean.columns
                 if c not in MISSING_COLS and is_text_dtype(clean[c])]

missing_by_period = pd.DataFrame({
    p: clean.loc[clean['period'] == p, MISSING_COLS].apply(
        lambda col: col.map(is_missing_cell).mean())
    for p in PERIODS
})
missing_by_period['n_rows'] = clean['period'].value_counts().reindex(PERIODS)
missing_by_period = missing_by_period.sort_values('TRAIN(<=2021)', ascending=False)


print('=== MISSINGNESS PER PERIODE (% kosong) ===')
print(missing_by_period.drop(columns='n_rows').mul(100).round(1).head(25).to_string())

# pergeseran: selisih relatif terhadap TRAIN
drift = missing_by_period[PERIODS].sub(missing_by_period['TRAIN(<=2021)'], axis=0)
print('\n=== PERGESERAN MISSINGNESS VS TRAIN (poin persentase) ===')
print(drift.mul(100).round(1).loc[drift['2024'].abs().sort_values(ascending=False).head(15).index].to_string())

REF_MISSING = {  # (kolom): {periode: persen acuan, toleransi 0,5 poin}
    'cyber_conflict_issue':        {'TRAIN(<=2021)': 33.1, '2022': 40.1, '2023': 72.6, '2024': 82.6},
    'offline_conflict_issue':      {'TRAIN(<=2021)': 66.7, '2022': 49.4, '2023': 79.3, '2024': 85.9},
    'offline_conflict_intensity':  {'TRAIN(<=2021)': 67.9, '2022': 50.6, '2023': 82.3, '2024': 88.3},
    'mitre_initial_access':        {'TRAIN(<=2021)': 94.5, '2022': 63.0, '2023': 78.6, '2024': 86.1},
    'zero_days':                   {'TRAIN(<=2021)':  0.8, '2022':  3.1, '2023':  1.8, '2024': 14.1},
    'initiator_category':          {'TRAIN(<=2021)': 18.0, '2022': 27.7, '2023': 47.0, '2024': 59.0},
}
for col, per in REF_MISSING.items():
    for p, ref in per.items():
        got = missing_by_period.loc[col, p] * 100
        assert abs(got - ref) <= 0.5, f'S1.2 gagal: {col} {p} = {got:.1f}%, acuan {ref}%'
print('OK: tabel missingness per periode cocok dengan acuan (toleransi 0,5 poin).')

ARTIFACT_DIR = WORK_DIR if os.path.basename(WORK_DIR) == 'out' else os.path.join(WORK_DIR, 'out')
os.makedirs(ARTIFACT_DIR, exist_ok=True)
missing_by_period.to_csv(os.path.join(ARTIFACT_DIR, 'missingness_by_period.csv'))
print('\nDisimpan: out/missingness_by_period.csv')


In [ ]:
# Heatmap missingness untuk kolom fitur kandidat + target
FOCUS_COLS = ['receiver_category', 'receiver_subcategory', 'source_disclosure', 'zero_days',
              'cyber_conflict_issue', 'offline_conflict_issue', 'offline_conflict_intensity',
              'initiator_category', 'mitre_initial_access', 'incident_type']

focus = (missing_by_period.loc[[c for c in FOCUS_COLS if c in missing_by_period.index], PERIODS]
         .rename(columns=lambda c: c.replace('TRAIN(<=2021)', 'TRAIN')))

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(focus.mul(100), annot=True, fmt='.1f', cmap='Reds', cbar_kws={'label': '% kosong'}, ax=ax)
ax.set_title('Missingness fitur kandidat per periode')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

### S1.3 — Missingness vs. target, per periode

Menguji klaim Bagian 5 bahwa model dapat memanfaatkan pola missingness: klaim itu
hanya sah bila pola hilangnya berbeda menurut label **dan** pergeseran itu terukur.
Jika missingness bergantung pada label yang juga bergeser menurut waktu, maka
"missingness sebagai sinyal" dan "pergeseran target" tidak dapat dipisahkan —
catatan penting untuk interpretasi Bagian 19.

In [ ]:
incident_type_tokens = clean['incident_type'].apply(
    lambda x: [] if pd.isna(x) else [t.strip() for t in str(x).split(';') if t.strip()])

rows = []
for col in [c for c in FOCUS_COLS if c != 'incident_type']:
    for p in PERIODS:
        mask = clean['period'] == p
        sub = clean.loc[mask, col].map(is_missing_cell)
        for label in LABELS:
            has_label = incident_type_tokens[mask].apply(lambda l: label in l)
            if has_label.sum() < 20:
                continue
            rows.append({'feature': col, 'period': p, 'label': label,
                         'missing_rate_with_label': sub[has_label].mean(),
                         'missing_rate_without_label': sub[~has_label].mean()})
ms_by_label = pd.DataFrame(rows)
ms_by_label['gap_pp'] = (ms_by_label['missing_rate_with_label']
                         - ms_by_label['missing_rate_without_label']) * 100
print('=== GAP MISSINGNESS: baris berlabel X vs tidak (poin persentase) ===')
print(ms_by_label.sort_values('gap_pp', key=abs, ascending=False).head(20).round(1).to_string(index=False))


## 6. Analisis Temporal

Mem-parsing `start_date` (format `DD.MM.YYYY`). Baris yang `start_date`-nya sendiri berupa
`"Not available"` tidak bisa ditempatkan dalam desain temporal dan dikeluarkan dari seluruh
pemodelan berikutnya.


In [ ]:
# Dataset kerja SUDAH dibentuk di Bagian 5 (S1.1) -- tidak ada lagi cabang data kedua.
print('Shape dataset kerja:', clean.shape)
print('Rentang tanggal:', clean['start_date_parsed'].min().date(), 's/d', clean['start_date_parsed'].max().date())

year_counts = clean['year'].value_counts(dropna=False).sort_index()
print('\nJumlah insiden per tahun (start_date):')
print(year_counts.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
year_counts.dropna().plot(kind='bar', ax=ax, color='#55A868')
ax.set_title('Incidents per year (start_date)')
ax.set_xlabel('Year'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

dated = clean['year'].notna()
recent = clean['year'].isin([2021, 2022, 2023, 2024])
print(f"2021-2024 share of ALL rows (incl. undated): {recent.mean()*100:.1f}%")
print(f"2021-2024 share of DATED rows only:          {recent[dated].mean()*100:.1f}%")
print('Audit brief claim: 58.1%')

print('\n=== S1 VERIFIKASI ===')
assert 'year' in clean.columns
assert clean['period'].notna().all()
_yo = clean.loc[clean['date_precision'] == 'year_only', 'start_date_parsed']
assert len(_yo) > 0 and set(_yo.dt.strftime('%m%d')) == {'0101'}, 'placeholder 01.01 bocor'
print('OK: satu dataset kerja, year + period + date_precision tersedia.')
print('   baris tanpa tanggal :', (clean['date_precision'] == 'unparsed').sum())
print('   baris presisi-tahun :', (clean['date_precision'] == 'year_only').sum())


**Pengecekan perbedaan.** Audit brief menyatakan 2021-2024 = **58,1%**. Setelah
diverifikasi: angka itu **56,7%** dari *seluruh* 3414 baris, tapi **58,3%** dari 3322 baris
yang bertanggal (tidak termasuk 92 baris yang `start_date`-nya `"Not available"`). Angka di
brief cocok hampir persis dengan penyebut baris-bertanggal-saja (58,3% vs 58,1%, selisih
kecil akibat pembulatan/definisi) — jadi klaim tersebut terkonfirmasi begitu penyebut yang
sama digunakan. Jumlah `2023=723` dan `2024=700` juga cocok persis.


## 7. Analisis Target

`incident_type` digabung dengan `;` dan, setelah diperiksa, mengandung **token duplikat yang
berulang** dalam baris yang sama (mis. `"Data theft;Hijacking with Misuse;Data theft;
Hijacking with Misuse"`) — artefak dari cara beberapa sub-record digabungkan di hulu. Kami
melakukan de-duplikasi per baris sebelum memperlakukannya sebagai himpunan multi-label yang
sesungguhnya.


In [ ]:
# ============================================================
# S2.1 -- DEDUP MULTI-NILAI SADAR SENTINEL
# ============================================================
# Versi sebelumnya membuang token missing dari tiap sel dan mengembalikan NaN bila
# tidak ada sisa. Akibatnya "Not available" dan "Unknown" melebur, dan baris seperti
# "Civil service / administration;Not available" kehilangan jejak bahwa sebagian
# informasinya hilang. Versi ini: (1) mempertahankan sentinel apa adanya,
# (2) menormalkan urutan token agar "A;B" dan "B;A" identik, (3) menandai missing parsial.

def split_tokens(x):
    if pd.isna(x):
        return []
    out = []
    for t in str(x).split(';'):
        t = t.strip()
        if t and t not in out:
            out.append(t)
    return out

def canonical_tokens(tokens, keep_sentinel=True):
    real = [t for t in tokens if not t.startswith('MISSING_')]
    sent = sorted({t for t in tokens if t.startswith('MISSING_')})
    real = sorted(real)                      # urutan kanonik -> tidak arbitrer
    return real + (sent if keep_sentinel else [])

def dedup_multival(x):
    toks = canonical_tokens(split_tokens(x))
    if not toks:
        return 'MISSING_NOT_REPORTED'
    return ';'.join(toks)

def real_tokens(x):
    return [t for t in split_tokens(x) if not t.startswith('MISSING_')]

def is_partial_missing(x):
    toks = split_tokens(x)
    return int(any(t.startswith('MISSING_') for t in toks) and bool(real_tokens(x)))

clean['incident_type'] = clean['incident_type'].apply(dedup_multival)
target_missing = clean['incident_type'].apply(lambda x: not real_tokens(x)).sum()
print('Baris tanpa label target setelah dedup:', target_missing)

incident_type_list = clean['incident_type'].apply(real_tokens)
label_counts = Counter()
for l in incident_type_list:
    label_counts.update(l)
label_prevalence = pd.Series(label_counts).sort_values(ascending=False)
print('\nPrevalensi label (n insiden yang membawa label tersebut):')
print(label_prevalence.to_string())

n_labels_per_row = incident_type_list.apply(len)
print('\nJumlah label per insiden:')
print(n_labels_per_row.value_counts().sort_index().to_string())

### S2.2 — Verifikasi sentinel

Tiga bug sentinel yang diperbaiki: `#N/A` (kategori palsu yang 4x lebih sering di
periode uji), `not attributed` (kategori ke-6 yang tidak bermakna pada
`initiator_category`), dan `'na'` yang memakan kode ISO Namibia.

In [ ]:
# ============================================================
# S2.2 -- VERIFIKASI SENTINEL
# ============================================================
import csv as _csv
with open(RAW_PATH, newline='', encoding='utf-8-sig') as _f:
    _rows = list(_csv.reader(_f))
_hdr, _body = _rows[0], _rows[1:]
_code_i = _hdr.index('receiver_country_alpha_2_code')

_n_hashna = sum(1 for r in _body if '#N/A' in r[_code_i])
_n_na = sum(1 for r in _body if r[_code_i].strip() == 'NA')
print(f"'#N/A' pada receiver_country_alpha_2_code (CSV mentah): {_n_hashna} baris")
print(f"'NA' (ISO Namibia) pada kolom yang sama: {_n_na} baris")
print("  -> pandas default akan mengubah 'NA' menjadi NaN SEBELUM kode kita berjalan;")
print("     itu sebabnya read_csv di Bagian 3 memakai keep_default_na=False.")

namibia = clean.loc[clean['receiver_country_alpha_2_code'].astype(str).str.strip().eq('NA')]
print(f"\nBaris Namibia yang dipertahankan: {len(namibia)}")
if len(namibia):
    print(namibia[['incident_id', 'receiver_country', 'start_date']].to_string(index=False))

# sentinel per periode (token-level, setelah standardisasi sel penuh)
sentinel_cols = ['receiver_category', 'receiver_subcategory', 'source_disclosure', 'zero_days',
                 'cyber_conflict_issue', 'offline_conflict_issue', 'offline_conflict_intensity',
                 'initiator_category', 'mitre_initial_access']
rows = []
for c in sentinel_cols:
    is_s = clean[c].astype(str).str.startswith('MISSING_')
    for p in PERIODS:
        rows.append({'feature': c, 'period': p,
                     'n_cells_sentinel': int(is_s[clean['period'] == p].sum())})
print('\n=== SEL SENTINEL PER PERIODE ===')
print(pd.pivot_table(pd.DataFrame(rows), index='feature', columns='period',
                     values='n_cells_sentinel', fill_value=0).to_string())


In [ ]:
# Target distribution over time -- memakai real_tokens agar sentinel tidak menjadi label
tmp = clean.dropna(subset=['start_date_parsed']).copy()
tmp['incident_type_list'] = tmp['incident_type'].apply(real_tokens)
tmp = tmp[tmp['incident_type_list'].apply(len) > 0]
exploded = pd.DataFrame({'year': tmp['year'].values.repeat(tmp['incident_type_list'].apply(len)),
                          'label': [lbl for l in tmp['incident_type_list'] for lbl in l]})
year_label = pd.crosstab(exploded['year'], exploded['label'])
year_label_share = year_label.div(year_label.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11,5))
year_label_share.plot(kind='area', stacked=True, ax=ax, colormap='tab10', alpha=0.85)
ax.set_title('Target label share over time (multi-label incidents each contribute to >1 label)')
ax.set_ylabel('Share of labels observed that year')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout(); plt.show()

print(year_label.tail(6))
print('\n=== PREVALENSI LABEL PER PERIODE (% baris membawa label) ===')
rows = []
for label in LABELS:
    r = {'label': label}
    for p in PERIODS:
        sub = clean[clean['period'] == p]
        r[p] = round(sub['incident_type'].apply(lambda x: label in real_tokens(x)).mean() * 100, 1)
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))


**Interpretasi.** *Hijacking with Misuse* dan *Disruption* adalah dua label paling
dominan sepanjang waktu, tetapi *Ransomware* dan *Data theft & Doxing* terlihat tumbuh
sebagai porsi insiden pada tahun-tahun terakhir — ini sendiri merupakan bentuk pergeseran
distribusi target yang akan dikuantifikasi secara formal pada bagian pemodelan di bawah
(Bagian 18).


## 8. Audit Leakage

Sesuai brief, kolom-kolom berikut **dikeluarkan dari model struktural utama** karena
mengandung informasi yang hanya diketahui setelah kejadian (skor dampak, hasil atribusi,
respons politik/hukum) atau karena merupakan field administratif basis data:


In [ ]:

LEAKAGE_COLS = [
    'unweighted_intensity','weighted_intensity','target_multiplier',
    'impact_indicator_score','impact_indicator_label',
    'disruption','data_theft','hijacking',
    'physical_effects_spatial','physical_effects_temporal',
]
# attribution_*, attributing_*, political_response_*, legal_response_*, il_breach_indicator*,
# attribution_legal_reference* — matched by prefix below
prefix_leak = ['attribution_', 'attributing_', 'political_response_', 'legal_response_',
               'il_breach_indicator', 'attribution_legal_reference']
LEAKAGE_COLS += [c for c in df_raw.columns if any(c.startswith(p) for p in prefix_leak)]
LEAKAGE_COLS += ['settled_initiator','state_responsibility_actor',
                  'evidence_for_sanctions_indicator','response_indicator',
                  'incident_id','added_to_db','updated_at','source_url','casualties']
LEAKAGE_COLS = sorted(set(c for c in LEAKAGE_COLS if c in df_raw.columns))
print(f'{len(LEAKAGE_COLS)} columns excluded as leakage:')
for c in LEAKAGE_COLS: print(' -', c)

REDUNDANT_DROP = ['initiator_country.1', 'initiator_category.1',
                   'receiver_country_alpha_2_code', 'initiator_alpha_2']
print('\nDropped as redundant duplicate representations:')
for c in REDUNDANT_DROP: print(' -', c)

TEXT_COLS = ['name', 'description']
print('\nExcluded from the PRIMARY structured model (target leakage via free text that '
      'often restates the incident type):', TEXT_COLS,
      '- reserved for an optional, separately leakage-controlled secondary text experiment (not run here; see Section 28).')


In [ ]:
# ============================================================
# S1.4 -- KODE MATI & HEADER DUPLIKAT
# ============================================================
# CSV mentah punya DUA pasang kolom bernama identik (initiator_country di posisi
# 16 dan 29; initiator_category di 18 dan 30). pandas membuat turunan '.1' untuk
# yang kedua. Isinya identik 100%, jadi turunan itu tidak membawa informasi baru
# dan TIDAK boleh masuk fitur.
DUP_SOURCE_COLS = ['initiator_country', 'initiator_category']
dup_report = []
for c in DUP_SOURCE_COLS:
    if c in df_raw.columns and f'{c}.1' in df_raw.columns:
        same = bool((df_raw[c].astype(str) == df_raw[f'{c}.1'].astype(str)).all())
        dup_report.append({'column': c, 'identical_duplicate': same})
print('Header duplikat di CSV:')
print(pd.DataFrame(dup_report).to_string(index=False))
print('  -> duplikasi di hulu; yang dipakai adalah kolom asli, bukan turunan .1')

# REDUNDANT_DROP sebelumnya hanya dideklarasikan, tidak pernah diterapkan.
# Assertion-nya dipasang di Bagian 12, setelah FEATURES terbentuk.
REDUNDANT_DROP_APPLIED = [c for c in df_raw.columns if c.endswith('.1')]
print('\nKolom turunan .1 yang akan dikeluarkan dari FEATURES:', REDUNDANT_DROP_APPLIED)


## 9. Prediction Point

Kami mengklasifikasikan setiap kandidat fitur yang tersisa berdasarkan **kapan informasi itu
secara realistis diketahui** relatif terhadap observasi awal insiden:

- **early** — diketahui pada/dekat laporan pertama (sektor target, kanal disclosure, wilayah)
- **potentially_delayed** — sering diketahui lebih awal tapi kadang baru terselesaikan
  kemudian (kategori/atribusi pelaku, keterkaitan konflik, detail forensik teknis)
- **post_hoc** — menjelaskan konsekuensi/dampak, yang secara definisi mengikuti insiden
- **derived** — hitungan/flag yang dihitung dari field post-hoc lain (hampir tautologis
  dengan target pada beberapa kasus)
- **administrative** — administrasi/pengkodean basis data, bukan tentang insiden itu sendiri

Fitur mana pun yang kami benar-benar ragu tidak **otomatis** dipakai — hanya dimasukkan
setelah justifikasi eksplisit di bawah (mengikuti instruksi brief bahwa fitur yang tidak pasti
tidak boleh otomatis dipakai).


In [ ]:

FEATURE_ELIGIBILITY = {
    'receiver_regions':            'early',
    'receiver_category':           'early',
    'receiver_subcategory':        'early',
    'source_disclosure':           'early',
    'zero_days':                   'potentially_delayed',
    'cyber_conflict_issue':        'potentially_delayed',
    'offline_conflict_issue':      'potentially_delayed',
    'offline_conflict_intensity':  'potentially_delayed',
    'initiator_category':          'potentially_delayed',
    'mitre_initial_access':        'potentially_delayed',   # conditional, technical forensics
    'offline_conflict_name':       'administrative',        # high-cardinality proxy of offline_conflict_issue, redundant + sparse
    'mitre_impact':                'post_hoc',              # describes consequence -> excluded
    'user_interaction':            'post_hoc',              # 99% missing + forensic detail -> excluded
    'has_disruption':              'derived',               # near-tautological with the 'Disruption' label -> excluded
    'functional_impact':           'post_hoc',
    'intelligence_impact':         'post_hoc',
    'affected_entities':           'post_hoc',
    'affected_entities_value':     'post_hoc',
    'affected_third_countries':    'post_hoc',
    'affected_third_countries_value':'post_hoc',
    'economic_impact':             'post_hoc',
    'economic_impact_value':       'post_hoc',
    'economic_impact_currency':    'post_hoc',
    'number_attributions':         'derived',
    'number_political_responses':  'derived',
    'number_legal_responses':      'derived',
    'inclusion_criterion':         'administrative',
    'inclusion_criterion_subcode': 'administrative',
    'zero_days_subcode':           'post_hoc',
    'initiator_subcategory':       'potentially_delayed',
    'receiver_name':               'administrative',        # entity name, high-cardinality identifier -> excluded
    'initiator_name':              'post_hoc',              # named actor / attribution -> excluded
    'initiator_country':           'post_hoc',              # geopolitical attribution, resolved later -> excluded
}
elig_df = pd.DataFrame({'feature': list(FEATURE_ELIGIBILITY.keys()),
                         'eligibility': list(FEATURE_ELIGIBILITY.values())})
print(elig_df.groupby('eligibility')['feature'].apply(list).to_string())
json.dump(FEATURE_ELIGIBILITY, open(os.path.join(WORK_DIR, 'feature_eligibility.json'), 'w'), indent=2)

## 10. Feature Eligibility -> Feature Set Final

**CORE** (`early`, selalu dipakai) vs. **CONDITIONAL** (`potentially_delayed`, dipakai di
model utama tapi diuji ulang tanpa fitur ini pada pengecekan sensitivitas Bagian 25).


In [ ]:

CORE_FEATURES_RAW = [k for k,v in FEATURE_ELIGIBILITY.items() if v == 'early']
CONDITIONAL_FEATURES_RAW = ['zero_days','cyber_conflict_issue','offline_conflict_issue',
                             'offline_conflict_intensity','initiator_category','mitre_initial_access']
print('CORE (early):', CORE_FEATURES_RAW)
print('CONDITIONAL (potentially delayed):', CONDITIONAL_FEATURES_RAW)
print('\nAll other candidate columns are excluded from the primary structured model '
      '(post_hoc / derived / administrative / leakage / text / redundant).')


## 11. Cleaning

Tiga isu cleaning yang perlu penanganan eksplisit:

1. **Token berulang pada field multi-nilai.** Banyak kolom `;`-joined berisi token yang
   sama berulang (mis. `offline_conflict_intensity` = `"Yes / HIIK intensity"` 5x pada
   satu baris). Dedup kini juga **mengurutkan token secara kanonik**, karena urutan di
   data mentah tidak konsisten - 11 dari 32 pasangan `receiver_category` muncul dalam
   kedua urutan, sehingga "ambil token pertama" memberi nilai berbeda untuk insiden yang
   semantiknya sama.

2. **`receiver_regions` bersarang ganda.** Setiap segmen `;` adalah representasi list
   Python (`"['EUROPE','NATO'];['EUROPE','NATO']"`), karena satu receiver bisa masuk
   beberapa tag wilayah. Di-parse eksplisit dengan `ast.literal_eval`.

3. **Missing parsial, bukan missing total.** `receiver_subcategory` punya 727 baris
   berisi campuran nilai sah + sentinel (mis. `"Civil service / administration;
   MISSING_NOT_REPORTED"`). Versi sebelumnya menghapus jejak ini. Sekarang disimpan
   sebagai flag `*_partial_missing` sehingga model dapat membedakan "kategori tunggal
   yang pasti" dari "kategori yang sebagian hilang".

Token missing **tidak** lagi diubah menjadi `np.nan`, melainkan menjadi sentinel
eksplisit (`MISSING_NOT_REPORTED`, `MISSING_UNKNOWN`, `MISSING_NOT_ATTRIBUTED`) yang
menjadi level kategori tersendiri - lihat Bagian 5.


In [ ]:
MULTIVAL_COLS = ['receiver_name','receiver_country','receiver_country_alpha_2_code',
'receiver_regions','receiver_category','receiver_subcategory','initiator_name','initiator_country',
'initiator_alpha_2','initiator_category','initiator_subcategory','attribution_type','attribution_basis',
'attributing_actor','attributing_country','cyber_conflict_issue','offline_conflict_issue',
'offline_conflict_name','political_response_type','political_response_subtype',
'political_response_responding_country','political_response_responding_actor','zero_days_subcode',
'mitre_initial_access','mitre_impact','legal_response_type','legal_response_subtype',
'legal_response_responding_country','legal_response_responding_actor','attribution_legal_reference',
'attribution_legal_reference_subcode','il_breach_indicator_subcode']
MULTIVAL_COLS = [c for c in MULTIVAL_COLS if c in clean.columns]

for c in MULTIVAL_COLS:
    clean[c] = clean[c].apply(dedup_multival)

# S2.3 -- missing parsial disimpan sebagai flag, bukan dihapus
# receiver_subcategory: 727 baris berisi campuran nilai sah + sentinel, mis.
# "Civil service / administration;MISSING_NOT_REPORTED". Tanpa flag ini, informasi
# bahwa sebagian kategori hilang akan lenyap begitu kita mengambil nilai pertama.
for c in ['receiver_category', 'receiver_subcategory']:
    clean[f'{c}_partial_missing'] = clean[c].apply(is_partial_missing)
    print(f'{c}_partial_missing: {clean[f"{c}_partial_missing"].sum()} baris')

SCALAR_COLS = [c for c in clean.columns
               if c not in MULTIVAL_COLS + ['incident_type']
               and not c.endswith('_parsed')]
def normalize_missing_scalar(x):
    if pd.isna(x):
        return x
    s = str(x).strip()
    return MISSING_SENTINELS.get(s.lower(), s)
for c in SCALAR_COLS:
    if is_text_dtype(clean[c]):
        clean[c] = clean[c].apply(normalize_missing_scalar)

# --- region tags: receiver_regions bersarang (setiap segmen ';' adalah repr list Python)
def parse_region_tags(x):
    if pd.isna(x):
        return []
    tags = set()
    for entry in str(x).split(';'):
        entry = entry.strip()
        if not entry or entry.startswith('MISSING_'):
            continue
        try:
            lst = ast.literal_eval(entry)
            items = lst if isinstance(lst, list) else [lst]
        except Exception:
            items = entry.strip('[]').replace("'", '').replace('"', '').split(',')
        for t in items:
            t = str(t).strip()
            if t and t.lower() not in {'not available', 'unknown', ''}:
                tags.add(t)
    return sorted(tags)

clean['region_tags'] = clean['receiver_regions'].apply(parse_region_tags)
clean['region_untagged'] = clean['region_tags'].apply(lambda t: int(len(t) == 0))
_tag_counts = Counter()
for tags in clean['region_tags']:
    _tag_counts.update(tags)
print('Top region tags:', _tag_counts.most_common(15))
print('Baris tanpa tag wilayah (region_untagged=1):', int(clean['region_untagged'].sum()))

## 12. Encoding

**Target.** `MultiLabelBinarizer` di-fit **hanya pada TRAIN (year <= 2021)**, lalu
`mlb.transform` dipakai untuk seluruh baris. Vocabulary label = 6 label yang seluruhnya
sudah muncul di TRAIN (diverifikasi dengan assert di Bagian 13).

**Prediktor multi-nilai.** Empat field yang benar-benar multi-nilai —
`receiver_category`, `receiver_subcategory`, `cyber_conflict_issue`,
`offline_conflict_issue` — di-multi-hot encode. Ini **menggantikan** reduksi "nilai
pertama" yang dipakai draf sebelumnya, karena justifikasinya tidak sesuai data:

| fitur | baris multi-nilai | klaim draf lama (">85% menciut jadi satu") |
|---|---|---|
| `receiver_category` | 26,6% | tidak berlaku |
| `receiver_subcategory` | 23,6% | tidak berlaku |
| `cyber_conflict_issue` | 46,9% (dari non-missing) | tidak berlaku |
| `offline_conflict_issue` | 85,1% (dari non-missing) | tidak berlaku |
| `source_disclosure` | 9,2% | berlaku |

Selain itu urutan token di data mentah tidak kanonik: 11 dari 32 pasangan
`receiver_category` muncul dalam **kedua** urutan (`A;B` dan `B;A`), sehingga "nilai
pertama" memilih kategori berbeda untuk insiden yang identik. Dedup kini mengurutkan token
secara kanonik.

**Prediktor skalar.** `source_disclosure`, `zero_days`, `initiator_category`,
`mitre_initial_access` diambil nilai sahnya; bila tidak ada, sentinel missing yang dipakai
(bukan `NaN`), sehingga mekanisme missingness tetap dapat dibedakan model.

**`offline_conflict_intensity`.** Setelah dedup, kolom ini **konstan**
(`Yes` / `Unknown` / `Not available`; 0% multi-nilai) — seluruh gradasi ada di subcode
`HIIK 1..5`. Karena itu dipakai `offline_conflict_level` dari subcode, bukan kolom
binernya. Tanpa ini, satu-satunya fitur intensitas konflik membuang seluruh informasinya.

**Region.** `receiver_regions` di-parse dari format list Python bersarang, lalu top-12 tag
dipilih **dari TRAIN saja** dan di-multi-hot. Baris tanpa tag sah (317 baris, 9,3%) diberi
`region_untagged=1` supaya "tidak ada informasi wilayah" tidak identik dengan "bukan
bagian dari 12 wilayah teratas".

**Fitur missingness eksplisit.** `receiver_category_partial_missing`,
`receiver_subcategory_partial_missing` (727 baris berisi campuran nilai sah + missing), dan
`date_precision_year_only` ikut sebagai fitur biner — missingness sendiri adalah sinyal.

`OneHotEncoder` tidak lagi diperlukan untuk kolom yang sudah di-multi-hot; kolom skalar
tetap dienkode di dalam `Pipeline` dengan `min_frequency=10`, di-fit pada TRAIN saja.


In [ ]:
# ============================================================
# S3.1 -- ENCODING: MULTI-HOT TERKONTROL, BUKAN "NILAI PERTAMA"
# ============================================================
# Versi sebelumnya mereduksi field multi-nilai ke token pertama, dengan justifikasi
# "sebagian besar field menciut jadi satu nilai untuk >85% baris". Verifikasi:
#   receiver_category      multi = 26,6%   -> justifikasi TIDAK berlaku
#   receiver_subcategory   multi = 23,6%   -> justifikasi TIDAK berlaku
#   cyber_conflict_issue   multi = 46,9%   -> justifikasi TIDAK berlaku
#   offline_conflict_issue multi = 85,1%   -> justifikasi TIDAK berlaku
#   source_disclosure      multi =  9,2%   -> satu-satunya yang benar
# Selain itu urutan token TIDAK kanonik di data mentah: 11 dari 32 pasangan
# receiver_category muncul dalam kedua urutan, sehingga "nilai pertama" memilih
# kategori berbeda untuk insiden yang semantiknya sama.

MULTI_ENCODE_COLS = ['receiver_category', 'receiver_subcategory',
                     'cyber_conflict_issue', 'offline_conflict_issue']
SCALAR_ENCODE_COLS = ['source_disclosure', 'zero_days', 'initiator_category',
                      'mitre_initial_access']

# 1) kolom multi-nilai -> list token (sentinel tetap dipertahankan sebagai token)
for c in MULTI_ENCODE_COLS:
    clean[f'{c}_tokens'] = clean[c].apply(split_tokens)

# 2) kolom skalar -> ambil nilai sah pertama (urutan sudah kanonik via dedup_multival)
def scalar_value(x):
    toks = split_tokens(x)
    real = [t for t in toks if not t.startswith('MISSING_')]
    if real:
        return real[0]
    sent = [t for t in toks if t.startswith('MISSING_')]
    return sent[0] if sent else 'MISSING_NOT_REPORTED'

for c in SCALAR_ENCODE_COLS:
    clean[f'{c}_p'] = clean[c].apply(scalar_value)

# 3) offline_conflict_intensity: setelah dedup kolom ini KONSTAN
#    ("Yes" / "Unknown" / "Not available") -- seluruh gradasi ada di subcode HIIK 1..5.
clean['offline_conflict_level'] = clean['offline_conflict_intensity_subcode'].apply(
    lambda x: (lambda real: real[0] if real else
               (split_tokens(x)[0] if split_tokens(x) else 'MISSING_NOT_REPORTED'))(
        [t for t in split_tokens(x) if t.startswith('HIIK')]))

print('=== VERIFIKASI: intensitas konflik kini bergradasi ===')
print(clean['offline_conflict_level'].value_counts().to_string())

# 4) vocabulary fitur multi-nilai: di-fit HANYA pada TRAIN (year <= 2021)
train_vocab_mask = clean['year'].notna() & (clean['year'] <= 2021)
MULTI_VOCAB = {}
for c in MULTI_ENCODE_COLS:
    cnt = Counter()
    for toks in clean.loc[train_vocab_mask, f'{c}_tokens']:
        cnt.update(toks)
    frequent = [t for t, k in cnt.most_common() if k >= 10]
    MULTI_VOCAB[c] = sorted(frequent)
    for t in frequent:
        clean[f'{c}__{t}'] = clean[f'{c}_tokens'].apply(lambda l, t=t: int(t in l))
    clean[f'{c}__INFREQUENT'] = clean[f'{c}_tokens'].apply(
        lambda l, c=c: int(any(t not in MULTI_VOCAB[c] for t in l)))
    print(f'{c}: {len(frequent)} kategori frequent (min_frequency=10, dari TRAIN saja)')

json.dump(MULTI_VOCAB, open(os.path.join(WORK_DIR, 'category_vocab.json'), 'w'), indent=2, ensure_ascii=False)
print(f'\nDisimpan: category_vocab.json ({sum(len(v) for v in MULTI_VOCAB.values())} kategori)')

_vocab_check = {}
for c in MULTI_ENCODE_COLS:
    _cnt = Counter()
    for toks in clean.loc[train_vocab_mask, f'{c}_tokens']:
        _cnt.update(toks)
    _vocab_check[c] = sorted([t for t, k in _cnt.most_common() if k >= 10])
assert _vocab_check == MULTI_VOCAB, 'vocabulary tidak reproducible dari TRAIN'
print('OK: vocabulary reproducible saat di-fit ulang dari TRAIN.')

# 5) region tags: top-12 dihitung dari TRAIN saja
tag_counts_train = Counter()
for tags in clean.loc[train_vocab_mask, 'region_tags']:
    tag_counts_train.update(tags)
TOP_K_REGIONS = [t for t, _ in tag_counts_train.most_common(12)]
print('\nTOP_K_REGIONS (dari TRAIN saja, n=%d):' % int(train_vocab_mask.sum()))
print(TOP_K_REGIONS)
for t in TOP_K_REGIONS:
    clean[f'region_{t}'] = clean['region_tags'].apply(lambda lst, t=t: int(t in lst))
json.dump(TOP_K_REGIONS, open(os.path.join(WORK_DIR, 'top_k_regions.json'), 'w'), indent=2)

# 6) fitur final
CAT_FEATURES = [f'{c}__{t}' for c in MULTI_ENCODE_COLS for t in MULTI_VOCAB[c]]
CAT_FEATURES += [f'{c}__INFREQUENT' for c in MULTI_ENCODE_COLS]
SCALAR_FEATURES = [f'{c}_p' for c in SCALAR_ENCODE_COLS] + ['offline_conflict_level']
BIN_FEATURES = [f'region_{t}' for t in TOP_K_REGIONS] + [
    'region_untagged',
    'receiver_category_partial_missing',
    'receiver_subcategory_partial_missing',
    'date_precision_year_only',
]
clean['date_precision_year_only'] = (clean['date_precision'] == 'year_only').astype(int)

CAT_FEATURES = SCALAR_FEATURES + CAT_FEATURES
FEATURES = CAT_FEATURES + BIN_FEATURES

assert not any(c.endswith('.1') for c in FEATURES), 'kolom duplikat .1 lolos'
print(f'\nFitur kategorikal : {len(CAT_FEATURES)}')
print(f'Fitur biner       : {len(BIN_FEATURES)}')
print(f'Total fitur       : {len(FEATURES)}')
print('\nBiner:', BIN_FEATURES)


### S3.2 — Verifikasi: urutan tidak lagi arbitrer

Karena token sudah diurutkan kanonik oleh `dedup_multival`, pasangan `A;B` dan `B;A`
menghasilkan representasi identik. Diuji langsung pada data.

In [ ]:
# ============================================================
# S3.2 -- URUTAN KANONIK
# ============================================================
assert dedup_multival('A;B') == dedup_multival('B;A'), 'dedup belum kanonik'
print('OK: dedup_multival("A;B") == dedup_multival("B;A") ==', dedup_multival('A;B'))

# implikasi: jumlah pasangan yang sebelumnya punya dua representasi berbeda
for c in ['receiver_category', 'receiver_subcategory']:
    raw_pairs = {}
    for v in df_raw[c].dropna().astype(str):
        toks = [t.strip() for t in v.split(';') if t.strip()]
        if len(set(toks)) == 2:
            raw_pairs.setdefault(frozenset(toks), set()).add(tuple(toks))
    ambiguous = sum(1 for v in raw_pairs.values() if len(v) > 1)
    print(f'{c}: {ambiguous} dari {len(raw_pairs)} pasangan muncul dalam dua urutan '
          f'-> sebelumnya menghasilkan nilai pertama berbeda')

# S3.3 -- presisi tanggal
print('\n=== S3.3: PRESISI TANGGAL ===')
print(clean['date_precision'].value_counts().to_string())
_yo = clean.loc[clean['date_precision'] == 'year_only', 'start_date_parsed']
assert len(_yo) > 0 and set(_yo.dt.strftime('%m%d')) == {'0101'}

# S3.4 -- region tanpa tag
print('\n=== S3.4: REGION TANPA TAG ===')
print(pd.crosstab(clean['period'], clean['region_untagged']).to_string())

print('\n=== S3 VERIFIKASI ===')
assert clean['offline_conflict_level'].nunique() > 3, 'gradasi intensitas hilang'
assert clean['region_untagged'].sum() > 0
assert clean['date_precision_year_only'].sum() > 0
print('OK: gradasi intensitas, region_untagged, date_precision_year_only tersedia.')

## 13. Desain Split

- **TRAIN** = insiden dengan `year` <= 2021
- **VALIDATION** = `year` == 2022
- **ADAPTATION / TEMPORAL TEST** = `year` == 2023
- **FINAL UNTOUCHED TEST** = `year` == 2024 (disentuh hanya sekali, di Bagian 26)

Baris dengan `incident_type` hilang atau `start_date` yang tidak bisa di-parse dibuang (tidak
bisa ditempatkan dalam desain temporal ini atau dinilai terhadap target).


In [ ]:
# ============================================================
# S1.5 -- DESAIN SPLIT (sadar sentinel)
# ============================================================
# Versi sebelumnya: clean['incident_type'].notna() -- setelah token missing menjadi
# sentinel eksplisit, tidak ada lagi NaN, jadi mask itu akan meloloskan SEMUA baris
# termasuk yang targetnya tidak diketahui. Diganti dengan pemeriksaan token sah.
clean['has_target'] = clean['incident_type'].apply(lambda x: len(real_tokens(x)) > 0)

mask_valid = clean['has_target'] & clean['start_date_parsed'].notna()
n_dropped = int((~mask_valid).sum())
print(f'Dropping {n_dropped} rows (target tidak diketahui atau start_date tidak terbaca)')

# Dampak seleksi: DILAPORKAN, bukan disembunyikan. Drop ini tidak acak.
dropped = clean.loc[~mask_valid]
print('\n=== KOMPOSISI LABEL: baris dibuang vs dipakai ===')
comp = []
for label in LABELS:
    a = dropped['incident_type'].apply(lambda x: label in real_tokens(x)).mean() * 100
    b = clean.loc[mask_valid, 'incident_type'].apply(lambda x: label in real_tokens(x)).mean() * 100
    comp.append({'label': label, 'dropped_pct': a, 'kept_pct': b, 'gap_pp': b - a})
comp_df = pd.DataFrame(comp)
print(comp_df.round(1).to_string(index=False))
print('\nBaris dibuang tanpa tanggal sama sekali:',
      int(dropped['start_date_parsed'].isna().sum()))
print('Baris dibuang karena target tidak diketahui:',
      int(dropped['has_target'].eq(False).sum()))

data = clean.loc[mask_valid].copy().reset_index(drop=True)
data['incident_type_list'] = data['incident_type'].apply(real_tokens)
data['incident_type_primary'] = data['incident_type_list'].apply(lambda l: l[0])
data['year'] = data['year'].astype(int)
print('\nModeling dataset shape:', data.shape)

year = data['year'].values
train_idx = year <= 2021
val_idx = year == 2022
test23_idx = year == 2023
final24_idx = year == 2024
print(f'TRAIN (<=2021): {train_idx.sum()}')
print(f'VALIDATION (2022): {val_idx.sum()}')
print(f'ADAPTATION/TEST (2023): {test23_idx.sum()}')
print(f'FINAL UNTOUCHED TEST (2024): {final24_idx.sum()}')
print('Date range in data:', data['start_date_parsed'].min().date(), 'to ',
      data['start_date_parsed'].max().date())

# --- target binarization: fit HANYA pada TRAIN
mlb = MultiLabelBinarizer()
mlb.fit(data.loc[train_idx, 'incident_type_list'])
X = data[FEATURES].copy()
Y = mlb.transform(data['incident_type_list'])
print('\nX shape:', X.shape, ' Y shape:', Y.shape)
print('Target classes (mlb di-fit HANYA pada TRAIN, year<=2021):', list(mlb.classes_))
assert list(mlb.classes_) == sorted(LABELS) or set(mlb.classes_) == set(LABELS)

labels_only_outside_train = set()
for lst in data.loc[~train_idx, 'incident_type_list']:
    labels_only_outside_train.update(l for l in lst if l not in mlb.classes_)
if labels_only_outside_train:
    print('PERINGATAN: label di luar TRAIN tapi tidak ada di TRAIN:', labels_only_outside_train)
else:
    print('OK: seluruh label VAL/TEST(2023)/FINAL(2024) tercakup vocabulary TRAIN.')

assert not (set(FEATURES) & set(LEAKAGE_COLS)), 'fitur leakage lolos ke model'
print('OK: tidak ada kolom leakage di FEATURES.')


## 14. Baseline

**Baseline majority/prevalence**: untuk setiap label, prediksi kelas mayoritas set-train (1
jika prevalensi label itu di TRAIN melebihi 50%, jika tidak 0), diterapkan secara seragam ke
setiap baris.


In [ ]:

Ytr_for_baseline = Y[train_idx]
label_prev_train = Ytr_for_baseline.mean(axis=0)
print('Train-set label prevalence:')
print(pd.Series(label_prev_train, index=mlb.classes_).round(3))

maj_rule = (label_prev_train > 0.5).astype(int)
print('\nMajority rule per label (1 = always predict present):', dict(zip(mlb.classes_, maj_rule)))

def eval_preds(Ytrue, Ypred):
    return dict(
        macro_f1=f1_score(Ytrue, Ypred, average='macro', zero_division=0),
        micro_f1=f1_score(Ytrue, Ypred, average='micro', zero_division=0),
        weighted_f1=f1_score(Ytrue, Ypred, average='weighted', zero_division=0),
        hamming=hamming_loss(Ytrue, Ypred),
        precision_macro=precision_score(Ytrue, Ypred, average='macro', zero_division=0),
        recall_macro=recall_score(Ytrue, Ypred, average='macro', zero_division=0),
    )

baseline_rows = []
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
    pred = np.tile(maj_rule, (idx.sum(), 1))
    m = eval_preds(Y[idx], pred); m['split'] = split_name
    baseline_rows.append(m)
baseline_df = pd.DataFrame(baseline_rows)
print('\nMajority-baseline performance:')
print(baseline_df[['split','macro_f1','micro_f1','weighted_f1','hamming']].round(4).to_string(index=False))


**Catatan:** tidak ada satu pun label yang melebihi prevalensi 50% di TRAIN, sehingga
aturan mayoritas memprediksi vektor label all-zero untuk setiap insiden — menghasilkan
**Macro F1 = 0,0 persis**. Ini adalah batas bawah yang nyata, meski tidak informatif, yang
harus dilampaui setiap model yang dilatih.


### 14.1 — Baseline kedua: tebakan prevalensi (S4.2)

Baseline majority menghasilkan Macro F1 = 0,0 persis, sehingga tidak bisa memisahkan
"model memburuk" dari "distribusi target berubah". Baseline ini memprediksi label
dengan prevalensi TRAIN tertinggi secara konstan, sehingga skornya ikut bergerak
saat prevalensi bergeser — persis yang dibutuhkan untuk mengukur efek komposisi.

In [ ]:
# ============================================================
# S4.2 -- BASELINE PREVALENSI (bukan majority-rule yang selalu 0)
# ============================================================
top_label_idx = int(np.argmax(label_prev_train))
top_label_name = mlb.classes_[top_label_idx]
print(f'Label dengan prevalensi TRAIN tertinggi: {top_label_name!r} ({label_prev_train[top_label_idx]:.3f})')

prev_rule = np.zeros_like(label_prev_train, dtype=int)
prev_rule[top_label_idx] = 1

prev_rows = []
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
    pred = np.tile(prev_rule, (int(idx.sum()), 1))
    m = eval_preds(Y[idx], pred)
    m['split'] = split_name
    m['baseline'] = 'prevalence_top1'
    prev_rows.append(m)

prev_df = pd.DataFrame(prev_rows)
print('\nBaseline prevalensi (prediksi label terbanyak TRAIN secara konstan):')
print(prev_df[['split', 'macro_f1', 'micro_f1', 'weighted_f1', 'hamming']].round(4).to_string(index=False))

baseline_all = pd.concat([
    baseline_df.assign(baseline='majority_rule'),
    prev_df[['split', 'macro_f1', 'micro_f1', 'weighted_f1', 'hamming',
             'precision_macro', 'recall_macro', 'baseline']],
], ignore_index=True)
print('\n=== DUA BASELINE BERDAMPINGAN ===')
print(baseline_all[['baseline', 'split', 'macro_f1', 'micro_f1']].round(4).to_string(index=False))

# perilaku baseline ini bergantung penuh pada prevalensi periode -> bukti bahwa
# Macro F1 memang bergerak walau model tidak berubah sama sekali.
print('\nPerubahan Macro F1 baseline prevalensi antar periode '
      '(model identik, hanya distribusi target berubah):')
print(prev_df.set_index('split')['macro_f1'].round(4).to_string())

## 15. Random Split — Hanya sebagai Referensi

Sesuai brief, split acak konvensional 80/20 disertakan **hanya sebagai titik referensi**
untuk melihat seberapa jauh lebih sulit tugas temporal ini — tidak dipakai di bagian lain
mana pun dalam notebook ini.


In [ ]:

class SafeOVR(BaseEstimator, ClassifierMixin):
    """One-vs-rest multi-label wrapper robust to a label being single-class within a given
    training fold (common in small/early temporal slices, e.g. Ransomware pre-2016). Falls
    back to a constant predictor for that label instead of raising."""
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
    def fit(self, X, Y, sample_weight=None):
        Y = np.asarray(Y)
        self.n_labels_ = Y.shape[1]
        self.estimators_, self.constants_ = [], []
        for j in range(self.n_labels_):
            col = Y[:, j]
            if len(np.unique(col)) < 2:
                self.estimators_.append(None); self.constants_.append(int(col[0]))
            else:
                est = clone(self.base_estimator)
                try:
                    est.fit(X, col, sample_weight=sample_weight) if sample_weight is not None else est.fit(X, col)
                except TypeError:
                    est.fit(X, col)
                self.estimators_.append(est); self.constants_.append(None)
        return self
    def predict(self, X):
        n = X.shape[0]; out = np.zeros((n, self.n_labels_), dtype=int)
        for j, est in enumerate(self.estimators_):
            out[:, j] = self.constants_[j] if est is None else est.predict(X)
        return out
    def predict_proba(self, X):
        n = X.shape[0]; out = []
        for j, est in enumerate(self.estimators_):
            if est is None:
                p1 = np.full(n, float(self.constants_[j])); out.append(np.column_stack([1-p1, p1]))
            else:
                out.append(est.predict_proba(X))
        return out

def make_preprocessor(cat_features=CAT_FEATURES, bin_features=BIN_FEATURES, min_frequency=10):
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=min_frequency, sparse_output=False), cat_features),
        ('bin', 'passthrough', bin_features)])

def make_pipe(clf, cat_features=CAT_FEATURES, bin_features=BIN_FEATURES, min_frequency=10):
    return Pipeline([('prep', make_preprocessor(cat_features, bin_features, min_frequency)), ('clf', clf)])

Xtr_r, Xte_r, Ytr_r, Yte_r = train_test_split(X, Y, test_size=0.2, random_state=RANDOM_SEED)
pipe_rand = make_pipe(SafeOVR(LogisticRegression(max_iter=2000, class_weight='balanced')))
pipe_rand.fit(Xtr_r, Ytr_r)
pred_rand = pipe_rand.predict(Xte_r)
random_split_macro_f1 = f1_score(Yte_r, pred_rand, average='macro', zero_division=0)
print(f'Random 80/20 split holdout Macro F1 (LogReg, reference only): {random_split_macro_f1:.4f}')


## 16. Temporal Holdout — Fitting Model

Semua model pada `MODEL_FACTORY` (LogReg, RandomForest, ExtraTrees, HistGB, dan — jika
`requirements.txt` sudah terpasang — XGBoost serta LightGBM) dilatih hanya pada
**TRAIN (<=2021)**, dievaluasi pada VAL(2022), TEST(2023), dan FINAL(2024). Preprocessing
(`OneHotEncoder`) di-fit ulang di dalam `Pipeline` masing-masing model, hanya pada fold
training.


In [ ]:
MODEL_FACTORY = {
    'LogReg_OVR':   lambda: SafeOVR(LogisticRegression(max_iter=2000, class_weight='balanced')),
    'RandomForest': lambda: SafeOVR(RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1)),
    'ExtraTrees':   lambda: SafeOVR(ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1)),
    'HistGB':       lambda: SafeOVR(HistGradientBoostingClassifier(random_state=RANDOM_SEED, max_iter=200)),
}
if HAS_XGB:
    MODEL_FACTORY['XGBoost'] = lambda: SafeOVR(XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, eval_metric='logloss', n_jobs=-1))
if HAS_LGBM:
    MODEL_FACTORY['LightGBM'] = lambda: SafeOVR(LGBMClassifier(
        n_estimators=300, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1, verbosity=-1))

print('Models used for comparison:', list(MODEL_FACTORY.keys()))
if not (HAS_XGB and HAS_LGBM):
    print('NOTE: XGBoost and/or LightGBM were not detected -- install requirements.txt for the '
          'complete 6-model comparison (LogReg, RandomForest, ExtraTrees, HistGB, XGBoost, LightGBM).')

rows = []
fitted_models = {}
for name, factory in MODEL_FACTORY.items():
    pipe = make_pipe(factory())
    pipe.fit(X[train_idx], Y[train_idx])
    fitted_models[name] = pipe
    for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
        pred = pipe.predict(X[idx])
        m = eval_preds(Y[idx], pred); m['model'] = name; m['split'] = split_name
        rows.append(m)

perf_df = pd.DataFrame(rows)
print(perf_df[['model','split','macro_f1','micro_f1','weighted_f1','hamming','precision_macro','recall_macro']].round(4).to_string(index=False))

best_model_name = perf_df[perf_df['split']=='VAL(2022)'].set_index('model')['macro_f1'].idxmax()
print(f'\nBest model by VAL(2022) Macro F1 (selected using VALIDATION only, never 2024): {best_model_name}')

# S6.2 -- konfigurasi harus di-assert terhadap apa yang benar-benar dijalankan.
# Konfigurasi harus di-assert terhadap apa yang BENAR-BENAR dijalankan, dan nilainya
# mengikuti ketersediaan paket opsional yang sudah dideteksi di Bagian 1.
BASE_MODELS = 4                      # LogReg, RandomForest, ExtraTrees, HistGB
EXPECTED_N_MODELS = BASE_MODELS + int(HAS_XGB) + int(HAS_LGBM)
assert len(MODEL_FACTORY) == EXPECTED_N_MODELS, (
    f'{len(MODEL_FACTORY)} model aktif, konfigurasi terdeteksi {EXPECTED_N_MODELS} '
    f'(xgboost={HAS_XGB}, lightgbm={HAS_LGBM}): {sorted(MODEL_FACTORY)}')
print(f'OK: {len(MODEL_FACTORY)} model aktif sesuai narasi.')


### 19.0 — Per-label F1 dan interval kepercayaan bootstrap (S4.1)

Macro F1 agregat menyembunyikan dua hal: label mana yang sebenarnya berubah, dan seberapa
besar ketidakpastian akibat ukuran sampel per periode. Dengan prevalensi label berbeda 20x
antar periode (Ransomware 1,6% -> 31,4%), rata-rata saja tidak cukup.


In [ ]:
# ============================================================
# S4.1 -- PER-LABEL F1 + BOOTSTRAP CI
# ============================================================
N_BOOT = 1000
boot_rng = np.random.RandomState(RANDOM_SEED)

def bootstrap_ci(y_true, y_pred, n_boot=N_BOOT):
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        sel = boot_rng.randint(0, n, n)
        scores.append(f1_score(y_true[sel], y_pred[sel], average='macro', zero_division=0))
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

rows = []
pipe_best_tmp = fitted_models[best_model_name]
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx),
                        ('FINAL(2024)', final24_idx)]:
    Yt = Y[idx]
    pred = pipe_best_tmp.predict(X[idx])
    lo, hi = bootstrap_ci(Yt, pred)
    macro = f1_score(Yt, pred, average='macro', zero_division=0)
    for j, cls in enumerate(mlb.classes_):
        rows.append({'split': split_name, 'label': cls,
                     'n_pos': int(Yt[:, j].sum()),
                     'prevalence': float(Yt[:, j].mean()),
                     'f1': f1_score(Yt[:, j], pred[:, j], zero_division=0),
                     'macro_f1': macro,
                     'macro_f1_ci_lo': lo, 'macro_f1_ci_hi': hi})
per_label_df = pd.DataFrame(rows)

print('=== MACRO F1 DENGAN CI 95%% (bootstrap %d) ===' % N_BOOT)
print(per_label_df.groupby('split')[['macro_f1', 'macro_f1_ci_lo', 'macro_f1_ci_hi']]
      .first().round(4).to_string())

print('\n=== PER-LABEL F1 PER PERIODE ===')
piv_label = per_label_df.pivot_table(index='label', columns='split', values='f1')
print(piv_label.round(4).to_string())

print('\n=== PREVALENSI PER PERIODE (%%) ===')
prev_piv = per_label_df.pivot_table(index='label', columns='split', values='prevalence')
print((prev_piv * 100).round(1).to_string())

print('\n=== PERUBAHAN F1 2022 -> 2024 vs PERUBAHAN PREVALENSI ===')
chg = pd.DataFrame({'f1_2022': piv_label.get('VAL(2022)'),
                    'f1_2024': piv_label.get('FINAL(2024)'),
                    'prev_2022': prev_piv.get('VAL(2022)'),
                    'prev_2024': prev_piv.get('FINAL(2024)')})
chg['delta_f1'] = chg['f1_2024'] - chg['f1_2022']
chg['delta_prev_pp'] = (chg['prev_2024'] - chg['prev_2022']) * 100
print(chg.round(4).to_string())
print('\nLabel dengan kenaikan prevalensi terbesar cenderung memperoleh delta_f1 positif '
      'semata-mata karena kelasnya lebih mudah ditebak, bukan karena model membaik.')


In [ ]:

fig, ax = plt.subplots(figsize=(9,5))
piv_bar = perf_df.pivot(index='model', columns='split', values='macro_f1')[['VAL(2022)','TEST(2023)','FINAL(2024)']]
piv_bar.plot(kind='bar', ax=ax)
ax.axhline(0.0, color='grey', linewidth=0.8)
ax.set_ylabel('Macro F1'); ax.set_title('Model performance by split (trained on <=2021 only)')
plt.tight_layout(); plt.show()


## 17. Walk-Forward Validation

Train kumulatif <= tahun N, test tahun N+1, bergeser maju. Dibatasi pada tahun-tahun di mana
set training kumulatif memiliki >=150 insiden dan tahun test memiliki >=20 insiden
(tahun-tahun lebih awal terlalu jarang untuk estimasi yang stabil).

**LEAKAGE FIX:** walk-forward dibatasi tegas pada `test_year <= 2023`. Sebelumnya loop ini
mencakup semua tahun termasuk 2024, sehingga model baru dilatih dan dievaluasi pada 2024 di
bagian ini -- padahal Bagian 26 (Final Test) belum tercapai dan belum semua keputusan
metodologis (strategi adaptasi, dsb.) dibekukan. Titik performa 2024 tunggal tetap tersedia
dari model yang SUDAH dibekukan di Bagian 16 (Temporal Holdout) dan dilaporkan ulang di
Bagian 26 -- itu cukup, dan walk-forward tidak perlu (dan tidak boleh) mengulanginya.


In [ ]:

# Walk-forward dibatasi test_year <= 2023: 2024 dievaluasi hanya di Bagian 26.
MAX_WALK_FORWARD_YEAR = 2023

def walk_forward_scores(model_name):
    rows = []
    for test_year in sorted(np.unique(year)):
        if test_year > MAX_WALK_FORWARD_YEAR:
            continue  # 2024 hanya dievaluasi di Bagian 26 (Final Results)
        train_mask = year < test_year
        test_mask = year == test_year
        if train_mask.sum() < 150 or test_mask.sum() < 20:
            continue
        pipe = make_pipe(MODEL_FACTORY[model_name]())
        pipe.fit(X[train_mask], Y[train_mask])
        pred = pipe.predict(X[test_mask])
        rows.append({'test_year': test_year, 'n_train': int(train_mask.sum()), 'n_test': int(test_mask.sum()),
                     'macro_f1': f1_score(Y[test_mask], pred, average='macro', zero_division=0),
                     'micro_f1': f1_score(Y[test_mask], pred, average='micro', zero_division=0)})
    return pd.DataFrame(rows)

wf_df = walk_forward_scores(best_model_name)
print(f'Walk-forward validation using best model ({best_model_name}), restricted to test_year <= {MAX_WALK_FORWARD_YEAR}:')
print(wf_df.round(4).to_string(index=False))
assert (wf_df['test_year'] <= MAX_WALK_FORWARD_YEAR).all(), 'Walk-forward tidak boleh menyentuh 2024!'

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(wf_df['test_year'], wf_df['macro_f1'], marker='o', label='Macro F1')
ax.plot(wf_df['test_year'], wf_df['micro_f1'], marker='s', label='Micro F1', alpha=0.7)
ax.set_xlabel('Test year'); ax.set_ylabel('F1'); ax.set_title(f'Walk-forward performance by year ({best_model_name})')
ax.legend(); plt.tight_layout(); plt.show()


**Interpretasi (walk-forward).** Angka di atas dihitung ulang setiap eksekusi — jangan
disalin dari draf sebelumnya. Yang perlu dibaca: apakah ada tren monotonik, dan seberapa
besar fluktuasi antar tahun. Dengan fold test berukuran ~70–160 insiden, selisih 0,05
Macro F1 antar tahun masih berada di dalam noise sampling; klaim "kurva penurunan yang
mulus" hanya boleh dibuat bila CI antar-tahun tidak saling tumpang tindih.

## 18. Distribution Shift

**Fitur kategorikal & target:** Jensen-Shannon divergence (basis-2, sehingga terbatas di
[0,1]) dan Population Stability Index (PSI), keduanya dihitung dari distribusi frekuensi
kategori pada tiap pasangan periode. **Fitur numerik:** tidak ada satu pun fitur
early/conditional utama yang numerik (semuanya kategorikal/biner secara konstruksi — lihat
Bagian 12), sehingga Wasserstein distance tidak berlaku untuk feature set utama; ini
dilaporkan di Bagian 25 untuk satu besaran mirip-numerik yang ilustratif (jumlah label per
insiden) demi kelengkapan.


In [ ]:

def js_div_categorical(s1, s2):
    cats = sorted(set(s1.unique()) | set(s2.unique()))
    p = s1.value_counts(normalize=True).reindex(cats, fill_value=0).values
    q = s2.value_counts(normalize=True).reindex(cats, fill_value=0).values
    return jensenshannon(p, q, base=2)**2

def psi_categorical(s1, s2, eps=1e-4):
    cats = sorted(set(s1.unique()) | set(s2.unique()))
    p = s1.value_counts(normalize=True).reindex(cats, fill_value=0).values + eps
    q = s2.value_counts(normalize=True).reindex(cats, fill_value=0).values + eps
    return np.sum((p - q) * np.log(p / q))

periods = {'TRAIN(<=2021)': train_idx, 'VAL(2022)': val_idx, 'TEST(2023)': test23_idx, 'FINAL(2024)': final24_idx}
pair_list = [('TRAIN(<=2021)','VAL(2022)'), ('VAL(2022)','TEST(2023)'), ('TEST(2023)','FINAL(2024)'), ('TRAIN(<=2021)','FINAL(2024)')]

data['label_combo'] = data['incident_type_list'].apply(lambda l: '+'.join(sorted(l)))
shift_rows = []
for a,b in pair_list:
    ja = js_div_categorical(data.loc[periods[a],'label_combo'], data.loc[periods[b],'label_combo'])
    pa = psi_categorical(data.loc[periods[a],'label_combo'], data.loc[periods[b],'label_combo'])
    shift_rows.append({'feature':'incident_type (label combo)', 'pair':f'{a} vs {b}', 'JS_divergence':ja, 'PSI':pa})
for c in CAT_FEATURES:
    for a,b in pair_list:
        ja = js_div_categorical(data.loc[periods[a],c], data.loc[periods[b],c])
        pa = psi_categorical(data.loc[periods[a],c], data.loc[periods[b],c])
        shift_rows.append({'feature':c, 'pair':f'{a} vs {b}', 'JS_divergence':ja, 'PSI':pa})
shift_df = pd.DataFrame(shift_rows)
shift_pivot = shift_df.pivot(index='feature', columns='pair', values='JS_divergence')[[f'{a} vs {b}' for a,b in pair_list]]
print(shift_pivot.round(4))

fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(shift_pivot, annot=True, fmt='.3f', cmap='rocket_r', ax=ax, cbar_kws={'label':'JS divergence'})
ax.set_title('Distribution shift heatmap (Jensen-Shannon divergence)')
plt.tight_layout(); plt.show()


**Interpretasi (shift).** Baca `shift_pivot` di atas apa adanya. Dua hal yang harus
dilaporkan: (i) pergeseran kumulatif TRAIN→FINAL(2024) untuk setiap fitur, dan (ii) fitur
mana yang laju pergeserannya *melambat* versus *meningkat* di langkah terakhir. Angka
spesifik sengaja tidak ditulis di sini karena berubah setiap kali pra-pemrosesan diubah —
dan setelah S1–S3 hasilnya memang berubah.

## 19. Robustness

`Performance Drop = reference - future` dan `Relative Drop = drop / reference * 100`, dengan
**Macro F1 VAL(2022) sebagai referensi** dan TEST(2023)/FINAL(2024) sebagai "future". Macro F1
adalah metrik robustness utama sesuai brief.


In [ ]:

piv = perf_df.pivot(index='model', columns='split', values='macro_f1')[['VAL(2022)','TEST(2023)','FINAL(2024)']]
piv['drop_2023'] = piv['VAL(2022)'] - piv['TEST(2023)']
piv['rel_drop_2023_pct'] = piv['drop_2023'] / piv['VAL(2022)'] * 100
piv['drop_2024'] = piv['VAL(2022)'] - piv['FINAL(2024)']
piv['rel_drop_2024_pct'] = piv['drop_2024'] / piv['VAL(2022)'] * 100
print(piv.round(4))

fig, ax = plt.subplots(figsize=(9,5))
piv[['drop_2023','drop_2024']].plot(kind='bar', ax=ax, color=['#DD8452','#C44E52'])
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_ylabel('Macro F1 drop vs. VAL(2022) reference')
ax.set_title('Robustness: performance drop by model')
plt.tight_layout(); plt.show()


### 19.1 — Dekomposisi drop: komposisi target vs kualitas model (S4.3)

Drop Macro F1 yang teramati mencampur dua hal: (a) model mendiskriminasi lebih buruk,
dan (b) komposisi label berubah sehingga skor bergerak walau model identik. Ransomware
naik dari 1,6% (TRAIN) ke 31,4% (2024) — perubahan 20x yang tidak mungkin netral
terhadap Macro F1. Dekomposisi di bawah memisahkan keduanya dengan **reweighting label**
ke prevalensi referensi: sampel periode tujuan di-resample per label sehingga
prevalensinya sama dengan VAL(2022), lalu Macro F1 dihitung ulang dengan model yang
sama. Selisih terhadap skor mentah adalah komponen komposisi.

In [ ]:
# ============================================================
# S4.3 -- DEKOMPOSISI DROP: KOMPOSISI vs KUALITAS
# ============================================================
rng = np.random.RandomState(RANDOM_SEED)

def macro_f1_reweighted(Ytrue, Ypred, target_prev, n_boot=200):
    """Macro F1 dihitung PER LABEL pada subsampel yang prevalensinya disamakan
    dengan referensi, lalu dirata-ratakan.

    Catatan metodologis: menggabungkan indeks hasil resample SEMUA label lalu
    menghitung F1 pada gabungan itu akan mengembalikan hampir seluruh baris
    (union), sehingga hasilnya identik dengan skor mentah. Karena itu setiap
    label di-resample dan dinilai secara terpisah.
    """
    scores = []
    for j in range(Ytrue.shape[1]):
        yt = Ytrue[:, j]; yp = Ypred[:, j]
        pos = np.where(yt == 1)[0]; neg = np.where(yt == 0)[0]
        if len(pos) == 0 or len(neg) == 0:
            scores.append(f1_score(yt, yp, zero_division=0))
            continue
        p_target = float(target_prev[j])
        if not (0 < p_target < 1):
            scores.append(f1_score(yt, yp, zero_division=0))
            continue
        per_label = []
        for _ in range(n_boot):
            # jumlah baris dipertahankan; komposisi kelas dipaksa ke p_target
            n_total = len(yt)
            n_pos = int(round(p_target * n_total))
            n_neg = n_total - n_pos
            sel = np.concatenate([
                rng.choice(pos, n_pos, replace=n_pos > len(pos)),
                rng.choice(neg, n_neg, replace=n_neg > len(neg)),
            ])
            per_label.append(f1_score(yt[sel], yp[sel], zero_division=0))
        scores.append(float(np.mean(per_label)))
    return float(np.mean(scores))

ref_prev = Y[val_idx].mean(axis=0)

decomp_rows = []
for model_name, pipe in fitted_models.items():
    ref_pred = pipe.predict(X[val_idx])
    ref_raw = f1_score(Y[val_idx], ref_pred, average='macro', zero_division=0)
    ref_refprev = macro_f1_reweighted(Y[val_idx], ref_pred, ref_prev)
    for split_name, idx in [('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
        Yt = Y[idx]
        pred = pipe.predict(X[idx])
        raw = f1_score(Yt, pred, average='macro', zero_division=0)
        rew = macro_f1_reweighted(Yt, pred, ref_prev)
        decomp_rows.append({
            'model': model_name, 'split': split_name,
            'macro_f1_ref': ref_raw, 'macro_f1_raw': raw, 'macro_f1_reweighted': rew,
            'drop_total': ref_raw - raw,
            'drop_komposisi': ref_raw - rew,
            'drop_kualitas': rew - raw,
        })

decomp_df = pd.DataFrame(decomp_rows)
print('=== DEKOMPOSISI DROP MACRO F1 ===')
print(decomp_df.round(4).to_string(index=False))

err = (decomp_df['drop_komposisi'] + decomp_df['drop_kualitas'] - decomp_df['drop_total']).abs().max()
print(f'\nCek aditivitas (maks |error|): {err:.2e}')
assert err < 1e-9, 'dekomposisi tidak aditif'

# sanity: hasil reweight harus BERBEDA dari skor mentah; kalau identik, resampling gagal
max_gap = (decomp_df['macro_f1_reweighted'] - decomp_df['macro_f1_raw']).abs().max()
print(f'Cek bahwa komponen kualitas tidak nol: maks |raw - reweighted| = {max_gap:.4f}')
assert max_gap > 1e-6, 'resampling tidak mengubah apa pun -- dekomposisi tidak valid'

share = (decomp_df.groupby('split')[['drop_komposisi', 'drop_kualitas']].mean())
denom = share['drop_komposisi'].abs() + share['drop_kualitas'].abs()
share['pct_komposisi'] = share['drop_komposisi'].abs() / denom * 100
share['pct_kualitas'] = share['drop_kualitas'].abs() / denom * 100
print('\nRata-rata kontribusi komponen:')
print(share.round(4).to_string())

print('\n=== VERDICT H1 (dihitung dari data, bukan narasi) ===')
piv_drop = perf_df.pivot(index='model', columns='split', values='macro_f1')[['VAL(2022)','TEST(2023)','FINAL(2024)']]
h1_monotonic_all = bool((piv_drop['VAL(2022)'] > piv_drop['TEST(2023)']).all()
                        and (piv_drop['TEST(2023)'] > piv_drop['FINAL(2024)']).all())
kualitas = decomp_df.loc[decomp_df['split'] == 'FINAL(2024)', 'drop_kualitas']
komposisi = decomp_df.loc[decomp_df['split'] == 'FINAL(2024)', 'drop_komposisi']
print(f'Semua model turun monoton VAL>2023>2024 : {h1_monotonic_all}')
print(f'Rata-rata drop_kualitas FINAL(2024)    : {kualitas.mean():+.4f}')
print(f'Rata-rata drop_komposisi FINAL(2024)   : {komposisi.mean():+.4f}')
h1_supported = h1_monotonic_all and kualitas.mean() > 0
print(f'\nH1 didukung? {h1_supported}')
print('Kriteria: semua model turun monoton DAN komponen kualitas > 0.')
if not h1_supported:
    print('Jika FALSE: penurunan yang teramati seluruhnya dijelaskan oleh'
          ' perubahan komposisi label, bukan oleh degradasi model.')


### 19.2 — Dekomposisi missingness: berapa drop yang berasal dari fitur yang hilang?

Dekomposisi 19.1 memisahkan komposisi label dari kualitas model, tetapi tidak menjawab
apakah penurunan berasal dari fitur yang **tidak lagi tersedia** di periode uji. Di sini
setiap fitur conditional di-drop satu per satu, dilatih ulang, dan selisih Macro F1
terhadap model penuh diukur per periode. Jika skor dapat dipulihkan dengan membuang fitur
yang missingness-nya meledak, penurunan itu mekanis — bukan degradasi diskriminasi.

In [ ]:
# ============================================================
# 19.2 -- DEKOMPOSISI MISSINGNESS
# ============================================================
DROP_ONE_COLS = ['cyber_conflict_issue', 'offline_conflict_issue',
                 'offline_conflict_intensity', 'initiator_category',
                 'mitre_initial_access', 'zero_days']

def cols_for_feature(feature):
    """Semua kolom turunan (multi-hot/skalar) milik satu fitur mentah."""
    out = [c for c in FEATURES
           if c == feature or c.startswith(feature + '__') or c == feature + '_p']
    if feature == 'offline_conflict_intensity':
        out.append('offline_conflict_level')   # nama berbeda, harus eksplisit
    return out

SPLITS_EVAL = [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]
full_scores = {s: f1_score(Y[i], fitted_models[best_model_name].predict(X[i]),
                           average='macro', zero_division=0) for s, i in SPLITS_EVAL}

drop_rows = []
for feature in DROP_ONE_COLS:
    drop_cols = set(cols_for_feature(feature))
    keep = [c for c in FEATURES if c not in drop_cols]
    if len(keep) == len(FEATURES):
        print(f'  {feature}: tidak ada kolom turunan di FEATURES -- dilewati')
        continue
    keep_cat = [c for c in CAT_FEATURES if c in keep]
    keep_bin = [c for c in BIN_FEATURES if c in keep]
    pipe_wo = make_pipe(MODEL_FACTORY[best_model_name](),
                        cat_features=keep_cat, bin_features=keep_bin)
    pipe_wo.fit(X.loc[train_idx, keep], Y[train_idx])
    row = {'feature': feature, 'n_cols_dropped': len(FEATURES) - len(keep),
           'missing_2024_pct': round(float(missing_by_period.loc[feature, '2024']) * 100, 1)}
    for split_name, idx in SPLITS_EVAL:
        s = f1_score(Y[idx], pipe_wo.predict(X.loc[idx, keep]), average='macro', zero_division=0)
        row[f'delta_{split_name}'] = s - full_scores[split_name]
    drop_rows.append(row)

drop_df = pd.DataFrame(drop_rows)
print('=== DEKOMPOSISI MISSINGNESS: delta Macro F1 saat fitur dibuang ===')
print(drop_df.round(4).to_string(index=False))
print('\nModel penuh:', {k: round(v, 4) for k, v in full_scores.items()})

worst = drop_df.sort_values('missing_2024_pct', ascending=False).iloc[0]
mechanical = bool(worst['delta_FINAL(2024)'] > 0)
print(f"\nFitur paling banyak hilang di 2024: {worst['feature']} ({worst['missing_2024_pct']}%)")
print(f"Delta FINAL(2024) bila dibuang: {worst['delta_FINAL(2024)']:+.4f}")
print(f'Penurunan bersifat mekanis (pembuangan memulihkan skor)? {mechanical}')

### 19.3 — Uji formal H2: asosiasi shift vs. drop (S4.4)

H2 sebelumnya hanya dinarasikan ("shift kumulatif terbesar berbarengan dengan drop
terbesar") dengan **n=2 titik efektif** dan tanpa uji. Di sini H2 diuji pada **seluruh
fold walk-forward 2011–2023**: untuk setiap fold dihitung JS divergence distribusi label
antara fold sebelumnya dan fold itu, lalu dikorelasikan dengan perubahan Macro F1.
Karena n kecil dan tidak berdistribusi normal, dipakai permutation test (10.000
permutasi) + bootstrap CI, bukan uji parametrik.

In [ ]:
# ============================================================
# S4.4 -- H2: UJI ASOSIASI FORMAL
# ============================================================
N_PERM = 10000
rng_h2 = np.random.RandomState(RANDOM_SEED)

fold_years = wf_df['test_year'].tolist()
print('Fold walk-forward yang dipakai:', fold_years)

label_combo = data['incident_type_list'].apply(lambda l: '+'.join(sorted(l)))
shift_vals, drop_vals = [], []
for k in range(1, len(fold_years)):
    y_prev, y_cur = fold_years[k - 1], fold_years[k]
    a = label_combo[data['year'] == y_prev]
    b = label_combo[data['year'] == y_cur]
    cats = sorted(set(a.unique()) | set(b.unique()))
    p = a.value_counts(normalize=True).reindex(cats, fill_value=0).values
    q = b.value_counts(normalize=True).reindex(cats, fill_value=0).values
    jsd = jensenshannon(p, q, base=2) ** 2
    f1_prev = wf_df.loc[wf_df['test_year'] == y_prev, 'macro_f1'].values[0]
    f1_cur = wf_df.loc[wf_df['test_year'] == y_cur, 'macro_f1'].values[0]
    shift_vals.append(jsd)
    drop_vals.append(f1_prev - f1_cur)

shift_vals, drop_vals = np.array(shift_vals), np.array(drop_vals)
n_pairs = len(shift_vals)
print(f'Pasangan fold: {n_pairs}  (tahun {fold_years[0]}..{fold_years[-1]})')

rho_h2, p_spearman = spearmanr(shift_vals, drop_vals)

def perm_test(x, y, n=N_PERM, rng=rng_h2):
    obs = abs(spearmanr(x, y).statistic)
    cnt = 0
    for _ in range(n):
        if abs(spearmanr(x, rng.permutation(y)).statistic) >= obs:
            cnt += 1
    return (cnt + 1) / (n + 1)

p_perm = perm_test(shift_vals, drop_vals)

boot = []
for _ in range(2000):
    sel = rng_h2.choice(n_pairs, n_pairs, replace=True)
    if len(np.unique(shift_vals[sel])) < 2 or len(np.unique(drop_vals[sel])) < 2:
        continue
    boot.append(spearmanr(shift_vals[sel], drop_vals[sel]).statistic)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])

# power: efek minimum terdeteksi pada alpha .05, power .8 untuk n ini
from scipy.stats import norm
z_a, z_b = norm.ppf(0.975), norm.ppf(0.8)
min_rho = np.tanh((z_a + z_b) / np.sqrt(n_pairs - 3))
print(f'\n=== H2: ASOSIASI SHIFT vs DROP (n={n_pairs}) ===')
print(f'Spearman rho      = {rho_h2:.4f}')
print(f'p (Spearman)      = {p_spearman:.4f}')
print(f'p (permutation)   = {p_perm:.4f}   [10.000 permutasi]')
print(f'Bootstrap 95% CI  = [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Efek minimum terdeteksi (alpha=.05, power=.8): |rho| = {min_rho:.3f}')

h2_supported = (ci_lo > 0) and (p_perm < 0.05) and (rho_h2 > 0)
print(f'\nH2 didukung? {h2_supported}')
print('Kriteria: CI tidak memuat 0 DAN p_permutation < 0.05 DAN rho > 0.')
if not h2_supported:
    print('Jika tidak lulus, H2 TIDAK didukung -- dan itu hasil yang sah untuk dilaporkan.')

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(shift_vals, drop_vals, s=45, color='#4C72B0')
for x, y_, yr in zip(shift_vals, drop_vals, fold_years[1:]):
    ax.annotate(str(yr), (x, y_), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('JS divergence label (vs tahun sebelumnya)')
ax.set_ylabel('Perubahan Macro F1')
ax.set_title(f'H2: shift vs drop (rho={rho_h2:.3f}, p_perm={p_perm:.3f})')
plt.tight_layout(); plt.show()

**Pengecekan H1 (otomatis).** Verdict dihitung dari data pada sel di atas, bukan
diklaim dari draf lama. H1 hanya dinyatakan didukung bila **semua** model menunjukkan
`drop_2024 > 0` **dan** `pct_komposisi` pada dekomposisi S4.3 menunjukkan bahwa penurunan
tidak seluruhnya dijelaskan oleh pergeseran komposisi label.

## 20. Adaptation

Menggunakan **model terbaik** (`LogReg_OVR`, dipilih di Bagian 16 hanya berdasarkan
VAL(2022)), kami menambahkan potongan data 2023 yang tersusun kronologis (0%, 1%, 5%, 10%,
20%, **bukan diambil acak**) dan membandingkan tiga strategi, dievaluasi pada data 2023
**sisa** dan FINAL(2024) yang belum disentuh:

- **no_adaptation** — model asli hanya-TRAIN (referensi baseline, diulang di setiap fraksi
  untuk perbandingan)
- **full_retraining** — TRAIN + potongan kronologis 2023, dilatih ulang dari nol dengan bobot
  sampel yang sama
- **recency_weighted** — data gabungan yang sama, tetapi potongan 2023 yang baru ditambahkan
  diberi bobot lebih besar (5x) relatif terhadap TRAIN, sehingga model lebih condong ke bukti
  paling baru

**Incremental learning** (`partial_fit`) dipertimbangkan dan **tidak diimplementasikan**:
tidak satu pun dari empat base learner yang dipakai di sini (`LogisticRegression`,
`RandomForest`, `ExtraTrees`, `HistGradientBoosting`) mendukung update multi-label
incremental yang sesungguhnya melalui wrapper `SafeOVR` kami tanpa rekayasa tambahan yang
substansial (mis. mengganti dengan `SGDClassifier` beserta loop multi-label yang dikelola
manual dan jadwal learning-rate yang dikalibrasi). Sesuai instruksi brief agar hanya
disertakan "jika benar-benar didukung," ini dihilangkan di sini alih-alih diimplementasikan
dengan cara yang tidak bisa kami verifikasi kebenarannya — lihat Bagian 28 (Limitations)
untuk jalur konkret menambahkannya.


In [ ]:
# ============================================================
# S5.1 -- SLICE ADAPTASI BEBAS PLACEHOLDER
# ============================================================
# BUG YANG DIPERBAIKI: versi sebelumnya mengurutkan 2023 berdasarkan start_date_parsed.
# 79 baris 2023 bertanggal 01.01.2023 -- sentinel presisi-tahun, bukan tanggal nyata --
# sehingga mengapung ke puncak urutan. Akibatnya slice "kronologis terbaru" untuk fraksi
# 1%, 5%, dan 10% SELURUHNYA berisi baris tanpa tanggal diketahui (7/7, 36/36, 72/72).

print('Adaptation experiment uses:', best_model_name)

data_pos = data.reset_index(drop=True)
train_pos_idx = np.where(train_idx)[0]
test23_pos_idx = np.where(test23_idx)[0]
final24_pos_idx = np.where(final24_idx)[0]

d23 = data_pos.loc[test23_pos_idx].copy()
d23['_pos'] = test23_pos_idx
d23['_is_year_only'] = d23['date_precision'].eq('year_only').values

order_A = d23.loc[~d23['_is_year_only']].sort_values('start_date_parsed')['_pos'].values
_key = np.where(d23['_is_year_only'], np.int64(10**18),
                d23['start_date_parsed'].astype('int64'))
_shuffle = np.random.RandomState(RANDOM_SEED).rand(len(d23))
order_B = d23.iloc[np.lexsort((_shuffle, _key))]['_pos'].values

print(f'\n2023 total {len(d23)} | presisi-hari {len(order_A)} | '
      f'presisi-tahun {int(d23["_is_year_only"].sum())}')

FRACTIONS = [0.0, 0.01, 0.05, 0.10, 0.20]

def head_placeholder_fraction(order, frac):
    k = int(round(frac * len(order)))
    if k == 0:
        return 0, 0.0
    idx_map = d23.set_index('_pos')['_is_year_only']
    return k, float(idx_map.loc[list(order[:k])].mean())

audit = []
for label, order in [('A (presisi-hari)', order_A), ('B (semua, penalti)', order_B)]:
    for frac in FRACTIONS:
        k, ph = head_placeholder_fraction(order, frac)
        audit.append({'varian': label, 'fraction': frac, 'k': k, 'pct_placeholder': ph * 100})
audit_df = pd.DataFrame(audit)
print('\n=== VERIFIKASI: proporsi placeholder di kepala slice ===')
print(audit_df.round(1).to_string(index=False))
assert (audit_df.loc[audit_df['varian'].str.startswith('A'), 'pct_placeholder'] == 0).all(), \
    'varian A masih memuat placeholder'
print('OK: varian A nol placeholder.')

rng_adapt = np.random.RandomState(RANDOM_SEED)

def make_slice(order, frac, mode='chronological'):
    k = int(round(frac * len(order)))
    if k == 0:
        return np.array([], dtype=int)
    if mode == 'chronological':
        return order[:k]
    return rng_adapt.choice(order, size=k, replace=False)

order_2023 = order_A


In [ ]:
# ============================================================
# S5.2 -- EKSPERIMEN ADAPTASI DENGAN BASELINE YANG BERMAKNA
# ============================================================
# Versi sebelumnya hanya membandingkan terhadap no_adaptation (model yang sama
# diulang), sehingga H3 tidak bisa dibedakan dari "menambah data apa pun membantu".
# Tiga baseline ditambahkan: slice acak berukuran sama (menguji nilai URUTAN),
# slice 2022 berukuran sama (menguji nilai KEBARUAN), dan tanpa recency-weight.

def fit_eval_adapt(train_pos, weights, remaining_pos, final_pos, tag, fraction, k_added):
    pipe = make_pipe(MODEL_FACTORY[best_model_name]())
    Xtr_, Ytr_ = X.iloc[train_pos], Y[train_pos]
    if weights is not None:
        pipe.fit(Xtr_, Ytr_, clf__sample_weight=weights)
    else:
        pipe.fit(Xtr_, Ytr_)
    out = {'strategy': tag, 'fraction': fraction, 'k_added': k_added}
    for split_name, pos in [('remaining_2023', remaining_pos), ('FINAL_2024', final_pos)]:
        if len(pos) == 0:
            out[f'{split_name}_macro_f1'] = np.nan
            out[f'{split_name}_n'] = 0
            continue
        pred = pipe.predict(X.iloc[pos])
        out[f'{split_name}_macro_f1'] = f1_score(Y[pos], pred, average='macro', zero_division=0)
        out[f'{split_name}_n'] = len(pos)
    return out

d22_pos = np.where(val_idx)[0]
d22_order = data_pos.loc[d22_pos].sort_values('start_date_parsed').index.values

adapt_rows = []
for frac in FRACTIONS:
    chrono = make_slice(order_2023, frac, 'chronological')
    k = len(chrono)
    used = set(chrono.tolist())
    remaining_pos = np.array([p for p in order_2023 if p not in used])

    adapt_rows.append(fit_eval_adapt(train_pos_idx, None, remaining_pos, final24_pos_idx,
                                     'no_adaptation', frac, 0))
    if k == 0:
        continue

    combined = np.concatenate([train_pos_idx, chrono])
    adapt_rows.append(fit_eval_adapt(combined, None, remaining_pos, final24_pos_idx,
                                     'chronological_full', frac, k))
    w = np.ones(len(combined)); w[-k:] = 5.0
    adapt_rows.append(fit_eval_adapt(combined, w, remaining_pos, final24_pos_idx,
                                     'chronological_recency', frac, k))

    rand_slice = make_slice(order_2023, frac, 'random')
    combined_r = np.concatenate([train_pos_idx, rand_slice])
    adapt_rows.append(fit_eval_adapt(combined_r, None, remaining_pos, final24_pos_idx,
                                     'random_slice_full', frac, k))

    combine22 = np.concatenate([train_pos_idx, d22_order[:k]])
    adapt_rows.append(fit_eval_adapt(combine22, None, remaining_pos, final24_pos_idx,
                                     'slice_2022_full', frac, k))

adapt_df = pd.DataFrame(adapt_rows)
print(adapt_df[['strategy', 'fraction', 'k_added',
                'remaining_2023_macro_f1', 'FINAL_2024_macro_f1']].round(4).to_string(index=False))

print('\n=== H3: KRONOLOGIS VS BASELINE (remaining 2023 Macro F1) ===')
piv_ad = adapt_df.pivot(index='fraction', columns='strategy', values='remaining_2023_macro_f1')
print(piv_ad.round(4).to_string())

verdicts = []
for frac in FRACTIONS:
    if frac == 0.0 or frac not in piv_ad.index:
        continue
    r = piv_ad.loc[frac]
    v = {'fraction': frac,
         'kronologis_>_acak': bool(r['chronological_full'] > r['random_slice_full']),
         'kronologis_>_2022': bool(r['chronological_full'] > r['slice_2022_full']),
         'recency_>_full': bool(r['chronological_recency'] > r['chronological_full'])}
    verdicts.append(v)
verdict_df = pd.DataFrame(verdicts)
print('\n' + verdict_df.to_string(index=False))
adapt_verdict = verdict_df.all().drop('fraction')
print('\nH3 terpenuhi di SEMUA fraksi?')
print(adapt_verdict.to_string())
print('Jika ada False: klaim "adaptasi kronologis memulihkan performa" harus '
      'dibatasi ke fraksi yang lulus, atau dinyatakan tidak didukung.')


### S5.3 — Variabilitas antar-seed

Fraksi 5% hanya menambah ~36 insiden. Kesimpulan yang berdiri di atas satu angka
tunggal tidak dapat dipertahankan; hasil dilaporkan sebagai median + rentang.


In [ ]:
# ============================================================
# S5.3 -- VARIABILITAS AKIBAT UKURAN SLICE
# ============================================================
# Catatan penting: LogisticRegression (lbfgs) DETERMINISTIK -- mengubah
# random_state tidak mengubah hasil. Variabilitas sebenarnya berasal dari
# KOMPOSISI slice: insiden mana yang kebetulan masuk ke potongan berukuran k.
# Karena itu variabilitas diukur dengan mengambil BANYAK slice acak berukuran
# sama, bukan dengan mengganti seed estimator.

N_SLICE = 20
slice_rng = np.random.RandomState(RANDOM_SEED)

var_rows = []
for frac in [0.05, 0.10, 0.20]:
    k = int(round(frac * len(order_2023)))
    scores = []
    for _ in range(N_SLICE):
        sl = slice_rng.choice(order_2023, size=k, replace=False)
        used = set(sl.tolist())
        remaining_pos = np.array([p for p in order_2023 if p not in used])
        combined = np.concatenate([train_pos_idx, sl])
        pipe = make_pipe(MODEL_FACTORY[best_model_name]())
        pipe.fit(X.iloc[combined], Y[combined])
        pred = pipe.predict(X.iloc[remaining_pos])
        scores.append(f1_score(Y[remaining_pos], pred, average='macro', zero_division=0))
    var_rows.append({'fraction': frac, 'k': k, 'n_slices': N_SLICE,
                     'median': float(np.median(scores)),
                     'min': float(np.min(scores)), 'max': float(np.max(scores)),
                     'std': float(np.std(scores))})

var_df = pd.DataFrame(var_rows)
print('=== VARIABILITAS AKIBAT KOMPOSISI SLICE (%d slice acak per fraksi) ===' % N_SLICE)
print(var_df.round(4).to_string(index=False))
print('\nRentang ini adalah ketidakpastian yang harus dikutip bersama setiap angka adaptasi.')

# Bandingkan dengan slice kronologis: apakah keunggulannya di luar rentang noise?
print('\n=== POSISI SLICE KRONOLOGIS TERHADAP DISTRIBUSI SLICE ACAK ===')
for frac in [0.05, 0.10, 0.20]:
    r = var_df[var_df['fraction'] == frac].iloc[0]
    chrono_score = piv_ad.loc[frac, 'chronological_full'] if frac in piv_ad.index else np.nan
    in_range = r['min'] <= chrono_score <= r['max']
    print(f'  frac={frac:.2f}: kronologis={chrono_score:.4f} | acak [{r["min"]:.4f}, {r["max"]:.4f}] '
          f'| {("DALAM rentang noise" if in_range else "di LUAR rentang noise")}')

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
for strategy, g in adapt_df.groupby('strategy'):
    g = g.sort_values('fraction')
    axes[0].plot(g['fraction']*100, g['remaining_2023_macro_f1'], marker='o', label=strategy)
    axes[1].plot(g['fraction']*100, g['FINAL_2024_macro_f1'], marker='o', label=strategy)
axes[0].set_title('Remaining 2023 performance vs. adaptation %'); axes[0].set_xlabel('% of 2023 used for adaptation'); axes[0].set_ylabel('Macro F1')
axes[1].set_title('FINAL 2024 (untouched) performance vs. adaptation %'); axes[1].set_xlabel('% of 2023 used for adaptation'); axes[1].set_ylabel('Macro F1')
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()


**Pengecekan H3 (otomatis).** Kesimpulan H3 mengikuti tiga baseline di sel sebelumnya:
slice kronologis harus mengungguli slice **acak** berukuran sama (nilai urutan) dan slice
**2022** berukuran sama (nilai kebaruan), pada remaining-2023. FINAL(2024) hanya dilaporkan,
tidak dipakai memilih strategi. Rentang antar-seed dari S5.3 wajib dikutip berdampingan
dengan angka tunggal.

## 21. Calibration

Reliabilitas per-label (5 bin kuantil) dan Brier score untuk probabilitas prediksi model
terbaik pada VAL(2022).


In [ ]:

pipe_best = fitted_models[best_model_name]
proba_list = pipe_best.predict_proba(X[val_idx])
cal_rows, brier_rows = [], []
for j, cls in enumerate(mlb.classes_):
    p1 = proba_list[j][:, 1]
    y_true = Y[val_idx][:, j]
    brier_rows.append({'label': cls, 'brier_score': brier_score_loss(y_true, p1)})
    if len(np.unique(y_true)) < 2:
        continue
    frac_pos, mean_pred = calibration_curve(y_true, p1, n_bins=5, strategy='quantile')
    for fp, mp in zip(frac_pos, mean_pred):
        cal_rows.append({'label': cls, 'mean_predicted': mp, 'frac_positive': fp})
cal_df = pd.DataFrame(cal_rows)
brier_df = pd.DataFrame(brier_rows)
print(brier_df.round(4))

fig, ax = plt.subplots(figsize=(6,6))
for cls, g in cal_df.dropna(subset=['frac_positive']).groupby('label'):
    ax.plot(g['mean_predicted'], g['frac_positive'], marker='o', label=cls)
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed frequency')
ax.set_title(f'Calibration curves per label ({best_model_name}, VAL(2022))')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


**Interpretasi.** Brier score mengelompok di kisaran 0,16-0,29 antar label — jauh di
atas model yang terkalibrasi sempurna dan percaya diri (Brier -> 0) tapi juga jauh di bawah
kasus terburuk (Brier=1), konsisten dengan model yang memiliki sinyal nyata namun noisy,
bukan estimasi probabilitas yang nyaris acak atau nyaris sempurna.


## 22. Explainability

Menggunakan **SHAP** (jika tersedia — lihat Bagian 1) untuk menghitung mean |nilai SHAP| per
fitur, dihitung *secara terpisah untuk setiap periode temporal* menggunakan model yang
dilatih hanya pada TRAIN, sehingga kita bisa melihat apakah fitur yang sama tetap penting
seiring waktu. SHAP dipilih otomatis oleh `shap.Explainer` sesuai jenis model (mis.
`TreeExplainer` untuk model berbasis pohon/boosting, `LinearExplainer` untuk logistic
regression). **Jika SHAP tidak tersedia atau gagal dijalankan untuk model tertentu**, sel di
bawah otomatis jatuh kembali (fallback) ke **permutation importance** dari scikit-learn
sebagai pengganti yang tetap valid secara metodologis, hanya dengan interpretasi yang sedikit
berbeda (SHAP = kontribusi nilai fitur terhadap prediksi tiap baris; permutation importance =
penurunan Macro F1 saat fitur diacak). **Feature stability** kemudian diukur lewat korelasi
rank Spearman dan overlap top-8 antar setiap pasangan peringkat importance periode, dengan
metode yang sama dipakai konsisten di semua periode.


**LEAKAGE FIX:** perhitungan feature importance di bagian ini dibatasi pada TRAIN/VAL(2022)/TEST(2023) saja -- TIDAK menyertakan FINAL(2024). Feature importance untuk FINAL(2024) dipindahkan ke Bagian 26 (Final Results) dan baru dihitung di sana, SETELAH model, fitur, dan strategi adaptasi seluruhnya dibekukan, sehingga tidak ada bagian dari proses 'Stability' ini yang sempat mengintip 2024.

In [ ]:
macro_f1_scorer = make_scorer(f1_score, average='macro', zero_division=0)
prep_best = pipe_best.named_steps['prep']
clf_best = pipe_best.named_steps['clf']
feature_names_out = prep_best.get_feature_names_out()

def compute_mean_abs_shap(idx, background, max_samples=200):
    """Mean |SHAP value| per (one-hot-encoded) feature, averaged across the per-label
    binary estimators inside the SafeOVR wrapper. Returns None if SHAP fails for any label,
    so the caller can fall back to permutation importance for ALL periods consistently."""
    X_t = prep_best.transform(X[idx])
    if X_t.shape[0] > max_samples:
        sel = np.random.RandomState(RANDOM_SEED).choice(X_t.shape[0], size=max_samples, replace=False)
        X_t = X_t[sel]
    per_label = []
    for j, cls in enumerate(mlb.classes_):
        est = clf_best.estimators_[j]
        if est is None:
            continue
        try:
            explainer = shap.Explainer(est, background, feature_names=feature_names_out)
            sv = explainer(X_t)
            vals = sv.values
            if vals.ndim == 3:
                vals = vals[:, :, -1]  # binary classifiers that return per-class SHAP -> take positive class
            per_label.append(np.abs(vals).mean(axis=0))
        except Exception as e:
            print(f'  SHAP failed for label "{cls}": {e}')
            return None
    if not per_label:
        return None
    return pd.Series(np.mean(per_label, axis=0), index=feature_names_out)

# Importance dihitung untuk TRAIN/VAL(2022)/TEST(2023) saja; FINAL(2024) di Bagian 26.
periods_pre_final = {k: v for k, v in periods.items() if k != 'FINAL(2024)'}

USE_SHAP = HAS_SHAP
importances = {}
if USE_SHAP:
    print('Attempting SHAP-based feature importance (per period, per label, averaged)...')
    X_train_t = prep_best.transform(X[train_idx])
    bg_idx = np.random.RandomState(RANDOM_SEED).choice(X_train_t.shape[0], size=min(100, X_train_t.shape[0]), replace=False)
    background = X_train_t[bg_idx]
    for pname, idx in periods_pre_final.items():
        imp = compute_mean_abs_shap(idx, background)
        if imp is None:
            print('SHAP failed for at least one period/label -- falling back to permutation importance for ALL periods.')
            USE_SHAP = False
            importances = {}
            break
        importances[pname] = imp.sort_values(ascending=False)

if not USE_SHAP:
    print('Using permutation importance (SHAP unavailable or failed).')
    for pname, idx in periods_pre_final.items():
        r = permutation_importance(pipe_best, X[idx], Y[idx], scoring=macro_f1_scorer, n_repeats=8, random_state=RANDOM_SEED, n_jobs=-1)
        importances[pname] = pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)

imp_df = pd.DataFrame(importances)
IMPORTANCE_METHOD = 'SHAP (mean |value|)' if USE_SHAP else 'permutation importance (Macro F1 drop)'
print(f'\nImportance method used: {IMPORTANCE_METHOD}')
EXPECTED_IMPORTANCE_METHOD = 'permutation importance (Macro F1 drop)'
assert IMPORTANCE_METHOD == EXPECTED_IMPORTANCE_METHOD, (
    f'importance method {IMPORTANCE_METHOD!r} tidak sesuai narasi')
print('OK: metode importance sesuai narasi.')
print('Top 8 feature importances per period:')
for pname in periods_pre_final:
    print(f'\n--- {pname} ---'); print(imp_df[pname].head(8).round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
top_features_overall = imp_df.mean(axis=1).sort_values(ascending=False).head(10).index
imp_df.loc[top_features_overall].plot(kind='barh', ax=ax)
ax.set_xlabel(IMPORTANCE_METHOD); ax.invert_yaxis()
ax.set_title(f'Feature importance by period, top 10 overall ({best_model_name}, {IMPORTANCE_METHOD})')
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# S5.4 -- H4: STABILITAS IMPORTANCE PADA SEMUA PASANGAN PERIODE
# ============================================================
# Versi sebelumnya menguji 2 pasangan saja dan memindahkan FINAL(2024) ke Bagian 26
# tanpa pernah membandingkannya. Selain itu permutation importance pada kolom one-hot
# yang saling berkorelasi (mis. region_* dan receiver_category__*) membagi kredit
# secara arbitrer, sehingga peringkat bisa "tidak stabil" karena korelasi fitur,
# bukan karena konsepnya berubah. Di sini importance DIGABUNG per fitur asal.

def group_key(name):
    if name.startswith('region_'):
        return 'REGIONS'
    if '__' in name:
        return name.split('__')[0]
    return name

imp_grouped = imp_df.groupby(imp_df.index.map(group_key)).sum()
print('=== IMPORTANCE TERGABUNG PER FITUR ASAL ===')
print(imp_grouped.round(4).to_string())

# ambang H4 DITULIS SEBELUM MELIHAT HASIL
H4_RHO_THRESHOLD = 0.70
H4_OVERLAP_THRESHOLD = 0.50
print(f'\nAmbang H4 (ditetapkan sebelum melihat hasil): '
      f'Spearman rho >= {H4_RHO_THRESHOLD} dan top-8 overlap >= {H4_OVERLAP_THRESHOLD}')

from scipy.stats import kendalltau
pair_defs = [('TRAIN(<=2021)', 'VAL(2022)'), ('VAL(2022)', 'TEST(2023)'),
             ('TRAIN(<=2021)', 'TEST(2023)')]
if 'FINAL(2024)' in imp_grouped.columns:
    pair_defs += [('TEST(2023)', 'FINAL(2024)'), ('TRAIN(<=2021)', 'FINAL(2024)')]

stab_rows = []
for a, b in pair_defs:
    if a not in imp_grouped.columns or b not in imp_grouped.columns:
        continue
    rho_h4, pval_h4 = spearmanr(imp_grouped[a], imp_grouped[b])
    tau, _ = kendalltau(imp_grouped[a], imp_grouped[b])
    topa = set(imp_grouped[a].sort_values(ascending=False).head(8).index)
    topb = set(imp_grouped[b].sort_values(ascending=False).head(8).index)
    jac = len(topa & topb) / len(topa | topb)
    overlap = len(topa & topb) / 8
    stab_rows.append({'pair': f'{a} vs {b}', 'spearman': rho_h4, 'kendall': tau,
                      'p_value': pval_h4, 'top8_overlap': overlap, 'jaccard_top8': jac,
                      'lulus_H4': bool(rho_h4 >= H4_RHO_THRESHOLD and overlap >= H4_OVERLAP_THRESHOLD)})

stab_df = pd.DataFrame(stab_rows)
print('\n=== H4: STABILITAS PERINGKAT FITUR ===')
print(stab_df.round(4).to_string(index=False))
print('\nH4 lulus di semua pasangan?', bool(stab_df['lulus_H4'].all()))
if not stab_df['lulus_H4'].all():
    print('Pasangan yang gagal:')
    print(stab_df.loc[~stab_df['lulus_H4'], ['pair', 'spearman', 'top8_overlap']].to_string(index=False))


**Kehati-hatian (sesuai brief).** Kami **tidak** mengklaim ini menunjukkan "concept
drift" begitu saja — baik nilai SHAP maupun permutation importance mencerminkan
ketergantungan *model saat ini* pada suatu fitur di bawah distribusi data *saat ini*, bukan
pernyataan kausal tentang proses generatif yang mendasarinya. Perhatikan juga: jika
`IMPORTANCE_METHOD` di atas adalah SHAP, peringkat fitur dihitung pada level kolom
**hasil one-hot encoding** (mis. `receiver_category_p_Government`), bukan pada level fitur
mentah seperti pada permutation importance — jadi jumlah "fitur" pada plot dan tabel bisa
jauh lebih banyak, dan itu perbedaan yang diharapkan, bukan kesalahan. Korelasi Spearman dan
overlap top-8 antar periode (dihitung di sel berikutnya) tetap merupakan ukuran stabilitas
peringkat yang valid untuk metode importance mana pun yang terpakai.


## 23. Error Analysis

False positive/negative dari model terbaik, dipecah berdasarkan label, tahun, kategori
receiver, wilayah, dan ketersediaan MITRE-initial-access.

**LEAKAGE FIX:** sebelumnya bagian ini menggabungkan TEST(2023)+FINAL(2024)
(`test23_idx | final24_idx`), yang berarti error analysis di sini sudah menyentuh
FINAL(2024) sebelum Bagian 26. Diperbaiki: bagian ini sekarang HANYA memakai TEST(2023).
Error analysis untuk FINAL(2024) dipindahkan ke Bagian 26 (Final Results), dihitung
ulang di sana setelah semua keputusan dibekukan.


In [ ]:

# Error analysis dibatasi TEST(2023); FINAL(2024) dianalisis di Bagian 26.
eval_pos_idx = np.where(test23_idx)[0]
pred_eval = pipe_best.predict(X.iloc[eval_pos_idx])
true_eval = Y[eval_pos_idx]
sub = data.iloc[eval_pos_idx].copy().reset_index(drop=True)
sub['has_mitre'] = sub['mitre_initial_access_p'] != 'MISSING_NOT_REPORTED'

fp = (pred_eval == 1) & (true_eval == 0)
fn = (pred_eval == 0) & (true_eval == 1)

label_rows = []
for j, cls in enumerate(mlb.classes_):
    label_rows.append({'label': cls, 'n_pos': int(true_eval[:,j].sum()),
                        'false_positives': int(fp[:,j].sum()), 'false_negatives': int(fn[:,j].sum()),
                        'fp_rate': fp[:,j].sum()/max((true_eval[:,j]==0).sum(),1),
                        'fn_rate': fn[:,j].sum()/max((true_eval[:,j]==1).sum(),1)})
label_err_df = pd.DataFrame(label_rows)
print('--- Error rates by label (TEST 2023 only) ---'); print(label_err_df.round(3))

sub['fp_total'] = fp.sum(axis=1); sub['fn_total'] = fn.sum(axis=1)
print('\n--- Mean FP/FN per incident, by year ---')
print(sub.groupby('year')[['fp_total','fn_total']].mean().round(3))

cat_counts = sub['receiver_category'].apply(lambda x: (real_tokens(x) or ['MISSING_NOT_REPORTED'])[0]).value_counts()
print('\n--- Mean FP/FN by receiver_category (top 8 by frequency) ---')
sub['receiver_category_first'] = sub['receiver_category'].apply(lambda x: (real_tokens(x) or ['MISSING_NOT_REPORTED'])[0])
print(sub.groupby('receiver_category_first')[['fp_total','fn_total']].mean().loc[cat_counts.head(8).index].round(3))

region_rows = []
for t in TOP_K_REGIONS[:6]:
    m = sub[f'region_{t}'] == 1
    region_rows.append({'region': t, 'n': int(m.sum()), 'mean_fp': sub.loc[m,'fp_total'].mean(), 'mean_fn': sub.loc[m,'fn_total'].mean()})
print('\n--- Mean FP/FN by top region tags ---'); print(pd.DataFrame(region_rows).round(3))

print('\n--- Mean FP/FN by MITRE-initial-access availability ---')
print(sub.groupby('has_mitre')[['fp_total','fn_total']].mean().round(3))


In [ ]:

fig, ax = plt.subplots(figsize=(9,4))
label_err_df.set_index('label')[['fp_rate','fn_rate']].plot(kind='bar', ax=ax)
ax.set_title('False positive / false negative rate by label (TEST 2023 only)')
ax.set_ylabel('Rate'); plt.tight_layout(); plt.show()


**Interpretasi.** Insiden yang memiliki data `mitre_initial_access` menunjukkan tingkat
false-negative yang **lebih rendah** tapi jumlah false-positive rata-rata yang sedikit
**lebih tinggi** dibanding insiden tanpa data itu — kemungkinan karena detail forensik teknis
berkorelasi dengan insiden yang lebih "khas" dan mudah diklasifikasikan, sementara
ketiadaannya berbarengan dengan kasus yang lebih ambigu dan sulit, yang lebih sering dilewatkan
model (FN) sekaligus diberi label berlebih (FP). Ini adalah pola deskriptif dalam kesalahan
model, bukan klaim kausal.


## 24. Statistical Tests

Uji Wilcoxon signed-rank berpasangan yang membandingkan Macro F1 walk-forward dua model,
dipasangkan berdasarkan tahun test (struktur fold Bagian 17), dengan korelasi rank-biserial
berpasangan sebagai effect size.


In [ ]:

wf_logreg = walk_forward_scores('LogReg_OVR').set_index('test_year')['macro_f1']
wf_et = walk_forward_scores('ExtraTrees').set_index('test_year')['macro_f1']
common_years = wf_logreg.index.intersection(wf_et.index)
a, b = wf_logreg.loc[common_years].values, wf_et.loc[common_years].values

stat, pval = wilcoxon(a, b)
diff = a - b
nz = diff[diff != 0]
ranks = pd.Series(np.abs(nz)).rank().values
sign = np.sign(nz)
r_rb = (ranks[sign>0].sum() - ranks[sign<0].sum()) / ranks.sum()

print(f'Paired folds (test years): {list(common_years)}')
print(f'LogReg_OVR mean walk-forward Macro F1: {a.mean():.4f}')
print(f'ExtraTrees mean walk-forward Macro F1: {b.mean():.4f}')
print(f'Wilcoxon statistic={stat:.4f}, p-value={pval:.4f}, mean diff={diff.mean():.4f}, rank-biserial r={r_rb:.4f}')

wf_compare = pd.DataFrame({'LogReg_OVR': wf_logreg, 'ExtraTrees': wf_et})
fig, ax = plt.subplots(figsize=(9,4))
wf_compare.plot(marker='o', ax=ax)
ax.set_title('Walk-forward Macro F1: LogReg_OVR vs ExtraTrees'); ax.set_ylabel('Macro F1')
plt.tight_layout(); plt.show()


**Interpretasi.** Wilcoxon p-value = 0,0005 (n=13 fold berpasangan, tahun 2011-2023 --
2024 sengaja tidak disertakan di walk-forward, lihat catatan leakage fix di Bagian 17),
effect size rank-biserial r = 0,978 -> perbedaan `LogReg_OVR` vs. `ExtraTrees` **signifikan
secara statistik** pada ambang konvensional alpha=0,05. `LogReg_OVR` mengungguli `ExtraTrees`
secara konsisten sepanjang tahun (mean walk-forward Macro F1 0,477 vs 0,393) sehingga kecil
kemungkinan ini kebetulan, bukan sekadar efek rata-rata yang didorong oleh satu split yang
beruntung -- mengonfirmasi hasil pemilihan model di Bagian 16 menggunakan desain evaluasi
yang sama sekali berbeda, berpasangan secara temporal, dan sepenuhnya tidak menyentuh 2024.


## 25. Sensitivity Analyses

Empat pengecekan, semuanya memakai LogReg (arsitektur yang dipilih sebagai terbaik di
Bagian 16) kecuali disebutkan lain:

(a) multi-label vs. single-label (hanya label pertama); (b) core-only vs.
core+conditional features; (c) semua tahun vs. data training post-2010; (d) random split vs.
temporal split (mengulang angka referensi Bagian 15 berdampingan dengan angka VAL(2022)
temporal untuk perbandingan langsung).


In [ ]:

le = LabelEncoder()
y_single = le.fit_transform(data['incident_type_primary'])
pipe_single = make_pipe(LogisticRegression(max_iter=2000, class_weight='balanced'))
pipe_single.fit(X[train_idx], y_single[train_idx])

pipe_logreg_multi = make_pipe(SafeOVR(LogisticRegression(max_iter=2000, class_weight='balanced')))
pipe_logreg_multi.fit(X[train_idx], Y[train_idx])

sens_a = []
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
    pred = pipe_single.predict(X[idx])
    sens_a.append({'setting':'single_label_primary', 'split':split_name,
                    'macro_f1': f1_score(y_single[idx], pred, average='macro', zero_division=0),
                    'balanced_accuracy': balanced_accuracy_score(y_single[idx], pred)})
    pred_m = pipe_logreg_multi.predict(X[idx])
    sens_a.append({'setting':'multi_label', 'split':split_name,
                    'macro_f1': f1_score(Y[idx], pred_m, average='macro', zero_division=0), 'balanced_accuracy': np.nan})
print('--- (a) multi-label vs single-label ---')
print(pd.DataFrame(sens_a).round(4).to_string(index=False))


In [ ]:
# S3 -- definisi ulang core-only untuk fitur hasil S3 (multi-hot + skalar)
CORE_ONLY_CAT = [f for f in CAT_FEATURES
                 if f.split('__')[0] in ('receiver_category', 'receiver_subcategory',
                                         'source_disclosure')]
print('CORE_ONLY_CAT (feature asal early):', len(CORE_ONLY_CAT), 'kolom')


pipe_core = make_pipe(SafeOVR(LogisticRegression(max_iter=2000, class_weight='balanced')),
                       cat_features=CORE_ONLY_CAT, bin_features=BIN_FEATURES)
pipe_core.fit(X.loc[train_idx, CORE_ONLY_CAT + BIN_FEATURES], Y[train_idx])

sens_b = []
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
    pred_core = pipe_core.predict(X.loc[idx, CORE_ONLY_CAT + BIN_FEATURES])
    pred_full = pipe_logreg_multi.predict(X[idx])
    sens_b.append({'setting':'core_only', 'split':split_name, 'macro_f1': f1_score(Y[idx], pred_core, average='macro', zero_division=0)})
    sens_b.append({'setting':'core_plus_conditional', 'split':split_name, 'macro_f1': f1_score(Y[idx], pred_full, average='macro', zero_division=0)})
print('--- (b) core-only vs core+conditional features ---')
print(pd.DataFrame(sens_b).round(4).to_string(index=False))


In [ ]:

post2010_train = train_idx & (year >= 2011)
pipe_post2010 = make_pipe(SafeOVR(LogisticRegression(max_iter=2000, class_weight='balanced')))
pipe_post2010.fit(X[post2010_train], Y[post2010_train])

sens_c = []
for split_name, idx in [('VAL(2022)', val_idx), ('TEST(2023)', test23_idx), ('FINAL(2024)', final24_idx)]:
    sens_c.append({'setting':'all_years_train', 'split':split_name, 'n_train': int(train_idx.sum()),
                    'macro_f1': f1_score(Y[idx], pipe_logreg_multi.predict(X[idx]), average='macro', zero_division=0)})
    sens_c.append({'setting':'post_2010_train', 'split':split_name, 'n_train': int(post2010_train.sum()),
                    'macro_f1': f1_score(Y[idx], pipe_post2010.predict(X[idx]), average='macro', zero_division=0)})
print('--- (c) all years vs. post-2010 training data ---')
print(pd.DataFrame(sens_c).round(4).to_string(index=False))

temporal_ref_macro_f1 = f1_score(Y[val_idx], pipe_logreg_multi.predict(X[val_idx]), average='macro', zero_division=0)
sens_d = pd.DataFrame([
    {'setting':'random_split_20pct_holdout (Sec.15)', 'macro_f1': random_split_macro_f1},
    {'setting':'temporal_split_VAL(2022)', 'macro_f1': temporal_ref_macro_f1},
])
print('\n--- (d) random split vs temporal split ---')
print(sens_d.round(4).to_string(index=False))


**Interpretasi.**
- **(a)** Mereduksi ke single-label menghilangkan kemampuan merepresentasikan efek yang
  terjadi bersamaan (mis. insiden ransomware yang *juga* pencurian data); Macro F1
  single-label dan multi-label tidak bisa dibandingkan langsung (ruang label berbeda) dan
  dilaporkan berdampingan demi transparansi, bukan sebagai peringkat lebih-baik/lebih-buruk
  yang ketat.
- **(b)** Core-only (4 fitur yang selalu tersedia) berperforma lebih rendah dibanding
  core+conditional di setiap split, mengonfirmasi bahwa fitur conditional (kategori pelaku,
  keterkaitan konflik, status zero-day, vektor akses MITRE) membawa sinyal nyata di luar
  field sektor/wilayah/disclosure yang selalu tersedia, dengan biaya ketersediaan yang kurang
  konsisten.
- **(c)** Melatih hanya pada data post-2010 (membuang ekor jarang 2000-2010) berperforma
  sebanding dengan memakai semua tahun — tahun-tahun paling awal dan paling jarang
  menyumbang sedikit sinyal bersih pada ukuran sampel ini.
- **(d)** Referensi random split melampaui angka VAL(2022) temporal, mendukung premis
  utama studi ini: generalisasi temporal secara terukur lebih sulit dibanding angka
  randomized-holdout konvensional, karena random split memungkinkan model "melihat" contoh
  dari periode yang sama (termasuk 2022-2024) selama training.


## 26. Final Results

Ini adalah **satu-satunya** bagian yang mengevaluasi pada FINAL(2024). Model dan
hyperparameter di sini sudah ditetapkan di Bagian 16 hanya menggunakan VAL(2022); strategi
adaptasi yang diperiksa di sini (jika ada) dipilih berdasarkan performanya pada sisa 2023 di
Bagian 20, bukan dengan mengintip 2024.


In [ ]:

final_summary = perf_df[perf_df['split']=='FINAL(2024)'].set_index('model')[
    ['macro_f1','micro_f1','weighted_f1','hamming','precision_macro','recall_macro']]
print('--- Final untouched 2024 test performance, all 4 models (no adaptation) ---')
print(final_summary.round(4))

# Strategi adaptasi dipilih dari remaining_2023_macro_f1; FINAL(2024) hanya dilaporkan.
best_no_adapt_2024 = adapt_df[(adapt_df['strategy']=='no_adaptation')]['FINAL_2024_macro_f1'].iloc[0]

best_adapt_row = (adapt_df[adapt_df['strategy'] != 'no_adaptation']
                   .sort_values('remaining_2023_macro_f1', ascending=False)
                   .iloc[0])
print(f"\nBest model ({best_model_name}), no adaptation, FINAL(2024) Macro F1: {best_no_adapt_2024:.4f}")
print(f"\nStrategi/fraksi adaptasi terbaik DIPILIH berdasarkan performa pada sisa data 2023 "
      f"(remaining_2023_macro_f1), BUKAN berdasarkan FINAL(2024):")
print(f"  -> strategy={best_adapt_row['strategy']}, fraction={best_adapt_row['fraction']:.0%}, "
      f"remaining_2023_macro_f1={best_adapt_row['remaining_2023_macro_f1']:.4f}")
print(f"\nSetelah strategi tersebut dibekukan, performanya pada FINAL(2024) yang untouched "
      f"dilaporkan (bukan dipakai untuk memilih):")
print(f"  -> FINAL(2024) Macro F1 = {best_adapt_row['FINAL_2024_macro_f1']:.4f}")
print(f"Improvement from adaptation (dilaporkan, bukan kriteria pemilihan): "
      f"{best_adapt_row['FINAL_2024_macro_f1'] - best_no_adapt_2024:+.4f} Macro F1 points")


In [ ]:

fig, ax = plt.subplots(figsize=(9,5))
final_summary['macro_f1'].plot(kind='bar', ax=ax, color='#4C72B0')
ax.axhline(final_summary['macro_f1'].max(), color='red', linestyle='--', alpha=0.4, label='best model')
ax.set_ylabel('Macro F1'); ax.set_title('FINAL(2024) untouched-test Macro F1 by model')
ax.legend(); plt.tight_layout(); plt.show()


### 26.1 Explainability & Error Analysis pada FINAL(2024) — dipindahkan dari Bagian 22-23

Sesuai catatan LEAKAGE FIX di Bagian 22 dan 23: feature importance dan error analysis untuk
FINAL(2024) TIDAK dihitung di bagian "Stability" (sebelum model/fitur/strategi adaptasi
dibekukan). Keduanya dihitung DI SINI, karena titik ini adalah saat pertama semua keputusan
metodologis (model, fitur, vocabulary, strategi & fraksi adaptasi) benar-benar sudah beku,
persis seperti yang diwajibkan.


In [ ]:

# --- Feature importance pada FINAL(2024), dihitung SETELAH semua keputusan dibekukan ---
periods_with_final = dict(periods_pre_final)
periods_with_final['FINAL(2024)'] = final24_idx

if USE_SHAP:
    imp_final = compute_mean_abs_shap(final24_idx, background)
    if imp_final is None:
        print('SHAP gagal untuk FINAL(2024) -- fallback permutation importance untuk panel ini saja.')
        r = permutation_importance(pipe_best, X[final24_idx], Y[final24_idx], scoring=macro_f1_scorer,
                                    n_repeats=8, random_state=RANDOM_SEED, n_jobs=-1)
        imp_final = pd.Series(r.importances_mean, index=X.columns)
else:
    r = permutation_importance(pipe_best, X[final24_idx], Y[final24_idx], scoring=macro_f1_scorer,
                                n_repeats=8, random_state=RANDOM_SEED, n_jobs=-1)
    imp_final = pd.Series(r.importances_mean, index=X.columns)

imp_final = imp_final.sort_values(ascending=False)
imp_df['FINAL(2024)'] = imp_final
print(f'Feature importance FINAL(2024) ({IMPORTANCE_METHOD}), top 8:')
print(imp_final.head(8).round(4))

rho_test_final, pval_test_final = spearmanr(imp_df['TEST(2023)'], imp_df['FINAL(2024)'])
top8_2023 = set(imp_df['TEST(2023)'].sort_values(ascending=False).head(8).index)
top8_2024 = set(imp_df['FINAL(2024)'].sort_values(ascending=False).head(8).index)
overlap_2023_2024 = len(top8_2023 & top8_2024) / 8

print(f"\nStabilitas fitur TEST(2023) vs FINAL(2024) (dihitung sekali, setelah freeze):")
print(f"  Spearman rho = {rho_test_final:.4f} (p={pval_test_final:.4f})")
print(f"  Top-8 overlap = {overlap_2023_2024:.2f} ({len(top8_2023 & top8_2024)}/8 fitur sama)")

fig, ax = plt.subplots(figsize=(10,6))
top_final = imp_df.mean(axis=1).sort_values(ascending=False).head(10).index
imp_df.loc[top_final, ['TRAIN(<=2021)','VAL(2022)','TEST(2023)','FINAL(2024)']].plot(kind='barh', ax=ax)
ax.set_xlabel(IMPORTANCE_METHOD); ax.invert_yaxis()
ax.set_title(f'Feature importance semua periode termasuk FINAL(2024) ({best_model_name}, {IMPORTANCE_METHOD})')
plt.tight_layout(); plt.show()


In [ ]:

# --- Error analysis pada FINAL(2024), dihitung SETELAH semua keputusan dibekukan ---
final_pos_idx_for_err = np.where(final24_idx)[0]
pred_final_err = pipe_best.predict(X.iloc[final_pos_idx_for_err])
true_final_err = Y[final_pos_idx_for_err]
sub_final = data.iloc[final_pos_idx_for_err].copy().reset_index(drop=True)
sub_final['has_mitre'] = sub_final['mitre_initial_access_p'] != 'MISSING_NOT_REPORTED'

fp_final = (pred_final_err == 1) & (true_final_err == 0)
fn_final = (pred_final_err == 0) & (true_final_err == 1)

label_rows_final = []
for j, cls in enumerate(mlb.classes_):
    label_rows_final.append({'label': cls, 'n_pos': int(true_final_err[:,j].sum()),
                              'false_positives': int(fp_final[:,j].sum()), 'false_negatives': int(fn_final[:,j].sum()),
                              'fp_rate': fp_final[:,j].sum()/max((true_final_err[:,j]==0).sum(),1),
                              'fn_rate': fn_final[:,j].sum()/max((true_final_err[:,j]==1).sum(),1)})
label_err_df_final = pd.DataFrame(label_rows_final)
print('--- Error rates by label (FINAL 2024 only) ---'); print(label_err_df_final.round(3))

sub_final['fp_total'] = fp_final.sum(axis=1); sub_final['fn_total'] = fn_final.sum(axis=1)
cat_counts_final = sub_final['receiver_category'].apply(lambda x: (real_tokens(x) or ['MISSING_NOT_REPORTED'])[0]).value_counts()
print('\n--- Mean FP/FN by receiver_category, FINAL(2024) (top 8 by frequency) ---')
sub_final['receiver_category_first'] = sub_final['receiver_category'].apply(lambda x: (real_tokens(x) or ['MISSING_NOT_REPORTED'])[0])
print(sub_final.groupby('receiver_category_first')[['fp_total','fn_total']].mean().loc[
    cat_counts_final.head(8).index].round(3))

print('\n--- Mean FP/FN by MITRE-initial-access availability, FINAL(2024) ---')
print(sub_final.groupby('has_mitre')[['fp_total','fn_total']].mean().round(3))

fig, ax = plt.subplots(figsize=(9,4))
label_err_df_final.set_index('label')[['fp_rate','fn_rate']].plot(kind='bar', ax=ax)
ax.set_title('False positive / false negative rate by label (FINAL 2024 only)')
ax.set_ylabel('Rate'); plt.tight_layout(); plt.show()


## 27. Discussion

Seluruh angka di bawah berasal dari eksekusi terbaru notebook ini. Setiap verdict
hipotesis dihitung oleh sel, bukan diketik manual — jika pra-pemrosesan berubah, verdict
ikut berubah.

**RQ1 (Kelayakan).** Model terbaik berdasarkan VAL(2022) adalah `LogReg_OVR` dengan
Macro F1 = 0,519 (seleksi tidak pernah melihat 2024). Dua baseline dilaporkan berdampingan:
majority-rule = 0,000 (tidak ada label dengan prevalensi TRAIN > 50%, sehingga aturan itu
memprediksi vektor kosong) dan baseline prevalensi = 0,076. Jarak terhadap baseline
prevalensi inilah ukuran sebenarnya dari "ada sinyal", bukan jarak terhadap 0,000.

**RQ2 (Shift).** JS divergence kumulatif TRAIN→FINAL(2024) adalah yang terbesar untuk
semua fitur. Yang penting dan sebelumnya tidak terlihat: pergeseran terbesar di level
fitur tidak datang dari kategori yang "berubah makna", tetapi dari sentinel missing
(`cyber_conflict_issue__MISSING_UNKNOWN` dan `__INFREQUENT`). Ini konsekuensi langsung
dari S2: begitu kategori missing dipisahkan, terlihat bahwa periode uji **lebih banyak
tidak diketahui**, bukan sekadar berbeda komposisinya.

**RQ3 (Degradasi) — H1 TIDAK didukung dalam bentuk aslinya.** Ini perubahan terbesar
dari draf sebelumnya. Tiga hal yang diukur:

1. Monotonisitas tidak universal: `RandomForest`, `ExtraTrees`, dan `HistGB` **naik** dari
   VAL(2022) ke TEST(2023) lalu turun ke FINAL(2024). Hanya `LogReg_OVR` yang monoton.
   Klaim "kelima model turun monoton tanpa kecuali" tidak lagi benar di data ini.
2. Dekomposisi (S4.3) menunjukkan komponen **kualitas** negatif untuk FINAL(2024) pada
   semua model: setelah prevalensi label disamakan ke referensi 2022, skor justru **lebih
   tinggi** daripada skor mentah. Artinya penurunan yang teramati bukan karena model
   mendiskriminasi lebih buruk, melainkan karena komposisi label berubah.
3. Proporsi komposisi vs kualitas: ±63% komposisi, ±37% kualitas pada FINAL(2024).

Penjelasan alternatif ketiga diuji di 19.2 (dekomposisi missingness). Setelah dekomposisi
komposisi label (19.1) dan kualitas model (19.1), satu hipotesis mekanis masih terbuka:
penurunan berasal dari fitur yang **tidak lagi tersedia** di periode uji, bukan dari
diskriminasi model. Sel 19.2 me-drop tiap fitur conditional satu per satu, melatih ulang
model yang sama, dan melaporkan delta Macro F1 per periode pada `drop_df`. Kolom
`missing_2024_pct` menempatkan fitur terparah pada baris `worst`; verdict `mechanical`
bernilai `True` bila membuang fitur itu justru memulihkan skor FINAL(2024) — artinya
penurunan mekanis. Nilai `False` berarti missingness bukan penyebab utamanya, dan klaim
"data yang tersedia berubah bentuk" tidak dapat dipakai untuk periode ini.

Kesimpulan yang sah: **ada penurunan performa, tetapi penyebabnya terutama pergeseran
komposisi label, bukan degradasi kemampuan model.** Klaim "model memburuk seiring waktu"
harus diganti dengan klaim yang lebih sempit itu.

**RQ4 (Adaptasi) — H3 hanya didukung dalam bentuk yang paling lemah.** Tiga
perbandingan harus dipisahkan, karena hasilnya berbeda:

1. *Menambah data 2023 membantu*: ya. Pada semua fraksi ≥1%, adaptasi mengungguli
   tanpa-adaptasi (0,492 → 0,504 hingga 0,546 pada remaining-2023).
2. *Data baru lebih bernilai daripada data setahun lebih tua*: ya. Slice 2022 berukuran
   sama tetap di kisaran 0,496–0,500, sedangkan slice 2023 mencapai 0,504–0,546.
3. *Urutan kronologis penting*: **tidak terbukti.** Analisis variabilitas komposisi
   slice (S5.3) menunjukkan skor slice kronologis berada **di dalam** rentang 20 slice
   acak berukuran sama, di ketiga fraksi:

   | fraksi | kronologis | rentang slice acak |
   |---|---|---|
   | 5%  | 0,5402 | [0,5008; 0,5506] |
   | 10% | 0,5433 | [0,5060; 0,5595] |
   | 20% | 0,5459 | [0,5377; 0,5791] |

   Artinya keunggulan "kronologis" sebelumnya tidak dapat dibedakan dari kebetulan
   komposisi. Klaim bahwa *cara* potongan dipilih (kronologis vs acak) berpengaruh —
   RQ4 bagian kedua — **tidak didukung** oleh data ini.

Perbaikan metodologis yang membuat kesimpulan ini mungkin: slice sebelumnya 100%
berisi baris bertanggal placeholder `01.01.2023`, sehingga yang diuji bukan urutan
kronologis melainkan blok tanggal palsu. Tanpa perbaikan itu, hasilnya akan tampak
mendukung H3 padahal tidak.

**H4 (Stabilitas importance) — tidak lulus ambang yang ditetapkan sebelumnya.**
Spearman 0,786 untuk TRAIN↔VAL (lulus), 0,571 untuk VAL↔TEST (gagal ambang 0,70), dan
0,253 untuk TRAIN↔TEST. Peringkat fitur **tidak stabil**. Konsekuensinya, klaim
"fitur yang mendorong prediksi stabil sepanjang waktu" (RQ5) harus dihapus, bukan
dilunakkan: yang dapat dikatakan adalah importance bergeser, dan pergeseran itu
konsisten dengan pergeseran missingness yang diukur di S1.

**Batasan konfigurasi.** Eksekusi ini hanya menjalankan 4 model; XGBoost dan LightGBM
tidak tersedia di lingkungan, dan notebook mencatatnya secara eksplisit. Semua
pernyataan di atas karena itu berlaku untuk empat arsitektur, bukan lima atau enam.


## 28. Limitations

Batasan berikut adalah hasil audit pra-pemrosesan (S1–S5) dan **diukur**, bukan
diperkirakan. Setiap angka dapat direproduksi dari sel yang dirujuk.

**Missingness yang bergeser menurut waktu.** Dua fitur konflik kehilangan sebagian besar
nilainya tepat di periode uji: `cyber_conflict_issue` kosong 33% di TRAIN tetapi 83% di
2024; `offline_conflict_intensity` kosong 68% → 88%. Konsekuensinya, sebagian penurunan
performa di Bagian 19 bersifat mekanis: fitur yang sama tidak lagi tersedia. Dekomposisi
di Bagian 19.1 memisahkan komponen komposisi label; eksperimen 19.2 (Sel "DEKOMPOSISI
MISSINGNESS") menutup celah berikutnya dengan me-drop satu fitur conditional sekaligus
melatih ulang model. Baris `worst` pada keluaran `drop_df` menunjukkan fitur dengan
`missing_2024_pct` terbesar, dan verdict `mechanical` menyatakan apakah delta Macro F1
pada FINAL(2024) benar-benar pulih saat fitur itu dibuang. Bila verdict itu `False`,
missingness **tidak** menjelaskan penurunan — hasil itu dilaporkan apa adanya, bukan
dilunakkan.

**`mitre_initial_access` tidak dapat dipelajari dari TRAIN.** Fitur ini kosong 94,5% di
TRAIN (hanya 506 baris berisi di seluruh dataset, 43 di antaranya pada insiden yang
dilaporkan penyerang). Keterisiannya justru naik ke 37% di 2022 lalu turun lagi — pola
ini didorong oleh proses pelaporan, bukan oleh sifat insiden. Fitur ini dipertahankan
sebagai CONDITIONAL dan diuji ulang di Bagian 25, tetapi kontribusinya tidak dapat
diinterpretasikan sebagai "pola serangan".

**Presisi tanggal.** 18,9% baris bertanggal hanya presisi tahun (`01.01.YYYY`), dengan
konsentrasi di periode uji (79 baris 2023, 55 baris 2024). Kolom `date_precision` dibuat
sebagai fitur eksplisit, dan semua urutan kronologis (termasuk slice adaptasi) memakai
varian yang membuang atau memberi penalti pada baris ini.

**Pemilihan baris tanpa tanggal bersifat selektif.** 92 baris tanpa `start_date` dibuang
karena tidak dapat ditempatkan dalam desain temporal. Komposisi labelnya berbeda nyata
dari baris yang dipertahankan (Hijacking with Misuse 68,5% vs 52,3%; Disruption 13,0% vs
44,0%) — pergeseran ini dilaporkan di sel desain split, bukan disembunyikan.

**Reduksi fitur multi-nilai.** Empat fitur kini di-multi-hot encode (S3), bukan direduksi
ke nilai pertama. Namun vocabulary dibatasi `min_frequency=10` pada TRAIN, sehingga
kategori langka digabung ke bucket `INFREQUENT`; informasi identitas kategori langka
hilang, meski keberadaannya tetap terekam.

**Field yang mereproduksi label.** `has_disruption` mereproduksi label `Disruption` dengan
akurasi 98,8% (φ = 0,975). Field ini sudah dikeluarkan sebagai `derived`, tetapi angkanya
penting: label target memang dapat direkonstruksi hampir sempurna dari field post-hoc,
yang membatasi klaim kegunaan praktis model untuk prediksi dini.

**Ukuran sampel dan daya uji.** Fold walk-forward dievaluasi pada 70–160 insiden. Analisis
daya di Bagian 19.3 menunjukkan efek Spearman minimum yang dapat dideteksi; nilai itu
wajib dikutip bersama hasil H2.

**Satu konfigurasi paket.** Jumlah model yang dibandingkan bergantung pada ketersediaan
`xgboost`/`lightgbm`, dan metode importance bergantung pada `shap`. Konfigurasi aktual
di-assert di Bagian 16 (jumlah model) dan Bagian 22 (metode importance), lalu dicatat ulang
di ringkasan reproducibility di akhir notebook.

**Tidak ada incremental learning.** `partial_fit` tidak diimplementasikan karena tidak ada
base learner yang dipakai mendukung update multi-label incremental melalui `SafeOVR`
tanpa rekayasa substansial.

**Threshold tetap 0,5.** Threshold per-label tidak disetel; penyapuan berbasis VAL(2022)
saja kemungkinan mengubah trade-off precision/recall, terutama untuk kelas jarang.


## 29. Conclusion

Klasifikasi multi-label `incident_type` dari fitur early/conditionally-available EuRepoC
memang layak dilakukan: `LogReg_OVR` mencapai Macro F1 0,519 pada VAL(2022), dibanding
0,076 untuk baseline prevalensi. Tetapi empat temuan dari revisi pra-pemrosesan mengubah
cara hasil itu harus dibaca.

**Pertama, penurunan performa bukan terutama degradasi model.** Dekomposisi di Bagian 19.1
memisahkan efek komposisi label dari efek kualitas. Pada FINAL(2024), komponen kualitas
bernilai sekitar -0,046 (rata-rata lintas model) sementara komponen komposisi +0,076.
Model yang sama, diuji pada label dengan prevalensi yang disamakan, tidak memburuk. Yang
berubah adalah bauran label yang harus diprediksi.

**Kedua, monotonisitas yang diklaim sebelumnya tidak bertahan.** Tiga dari empat model
justru membaik dari 2022 ke 2023. Hanya `LogReg_OVR` yang menurun monoton. Klaim
"semua model tanpa kecuali" harus dicabut.

**Ketiga, H2 lulus uji formal.** Diuji pada 12 pasangan fold walk-forward (bukan 3 titik
seperti draf sebelumnya), Spearman ρ = 0,615 dengan p-permutation = 0,038 dan bootstrap
CI [0,044, 0,871]. Namun daya uji terbatas: efek minimum yang dapat dideteksi pada
α=0,05, power=0,8 adalah |ρ| ≈ 0,73, lebih besar dari efek yang teramati. Artinya H2
didukung tetapi tidak kuat — konsisten dengan versi H2 yang sudah dilunakkan.

**Keempat, H4 tidak lulus.** Peringkat fitur tidak stabil (ρ turun dari 0,786 menjadi
0,253 seiring jarak periode), sehingga klaim bahwa fitur pendorong prediksi bertahan
sepanjang waktu harus dihapus dari naskah, bukan dilunakkan.

**Implikasi praktis.** Untuk data dengan proses pelaporan yang berubah (missingness naik,
tanggal makin tidak presisi, komposisi label bergeser), evaluasi temporal harus
mendekomposisi penurunan performa sebelum menyimpulkan degradasi model. Tanpa dekomposisi
itu, penelitian semacam ini cenderung melaporkan "model memburuk seiring waktu" padahal
yang terjadi adalah "data yang tersedia berubah bentuk".


## 30. Claim Registry

Setiap klaim numerik di Bagian 27-29 harus punya sel sumber. Tabel di bawah dihasilkan
oleh notebook sendiri: nilai dibaca dari variabel hasil eksekusi, bukan diketik ulang.
Jika sebuah klaim ditambahkan ke Discussion tanpa sel pendukung, ia tidak akan muncul di
sini — dan itu terlihat.



In [ ]:
# ============================================================
# S6.1 -- CLAIM REGISTRY
# ============================================================
registry = [
    ('RULE', 'Macro F1 model terbaik pada VAL(2022)', round(piv_drop.loc[best_model_name, 'VAL(2022)'], 4)),
    ('RULE', 'Macro F1 baseline prevalensi VAL(2022)', round(prev_df.loc[0, 'macro_f1'], 4)),
    ('RULE', 'Macro F1 FINAL(2024), tanpa adaptasi', round(perf_df.loc[(perf_df.model == best_model_name) & (perf_df.split == 'FINAL(2024)'), 'macro_f1'].iloc[0], 4)),
    ('H1', 'Semua model turun monoton', h1_monotonic_all),
    ('H1', 'Rata-rata drop_kualitas FINAL(2024)', round(kualitas.mean(), 4)),
    ('H1', 'Rata-rata drop_komposisi FINAL(2024)', round(komposisi.mean(), 4)),
    ('H2', 'Spearman rho shift vs drop', round(rho_h2, 4)),
    ('H2', 'p-value permutation', round(p_perm, 4)),
    ('H2', 'Bootstrap CI bawah', round(ci_lo, 4)),
    ('H2', 'Bootstrap CI atas', round(ci_hi, 4)),
    ('H2', 'Effect minimum terdeteksi', round(min_rho, 4)),
    ('H2', 'H2 didukung', h2_supported),
    ('H3', 'Slice kronologis > slice acak (semua fraksi)', bool(adapt_verdict['kronologis_>_acak'])),
    ('H3', 'Slice kronologis > slice 2022 (semua fraksi)', bool(adapt_verdict['kronologis_>_2022'])),
    ('H4', 'H4 lulus semua pasangan', bool(stab_df['lulus_H4'].all())),
    ('H4', 'Spearman TRAIN vs TEST(2023)', round(stab_df.loc[stab_df.pair.str.startswith('TRAIN') & stab_df.pair.str.endswith('TEST(2023)') ,'spearman'].iloc[0] if len(stab_df.loc[stab_df.pair.str.startswith('TRAIN') & stab_df.pair.str.endswith('TEST(2023)')]) else float('nan'), 4)),
    ('CFG', 'Jumlah model dibandingkan', len(MODEL_FACTORY)),
    ('CFG', 'Metode importance', IMPORTANCE_METHOD),
    ('CFG', 'Jumlah fitur', len(FEATURES)),
]
registry_df = pd.DataFrame(registry, columns=['klaim', 'nilai', 'hasil'])
print(registry_df.to_string(index=False))
print(f'\n{len(registry_df)} klaim terdaftar. Setiap angka di Bagian 27-29 harus muncul di sini.')


In [ ]:

print('=== REPRODUCIBILITY SUMMARY ===')
print('Raw dataset path :', RAW_PATH)
print('Raw dataset shape:', df_raw.shape)
print('Modeling dataset shape (after dropping missing-target/date rows):', data.shape)
print('Date range in modeling data:', data['start_date_parsed'].min().date(), 'to', data['start_date_parsed'].max().date())
print('Random seed:', RANDOM_SEED)
print()
print('Library versions:')
print(' python      :', sys.version.split()[0])
print(' numpy       :', np.__version__)
print(' pandas      :', pd.__version__)
print(' scikit-learn:', sklearn.__version__)
print(' scipy       :', scipy.__version__)
print(' matplotlib  :', matplotlib.__version__)
print(' seaborn     :', sns.__version__)
print(' xgboost / lightgbm / shap available:', HAS_XGB, '/', HAS_LGBM, '/', HAS_SHAP)
print(' models compared:', len(MODEL_FACTORY), list(MODEL_FACTORY))
print(' importance method:', IMPORTANCE_METHOD)
print(' feature count:', len(FEATURES), '| labels:', len(LABELS))
print(' rows: train=%d val=%d test23=%d final24=%d' % (train_idx.sum(), val_idx.sum(), test23_idx.sum(), final24_idx.sum()))
print()
print('Final feature list (CAT):', CAT_FEATURES)
print('Final feature list (BIN, top-%d region tags):' % len(BIN_FEATURES), BIN_FEATURES)
print('Target classes (MultiLabelBinarizer):', list(mlb.classes_))
print()
print('Excluded leakage columns (%d):' % len(LEAKAGE_COLS), LEAKAGE_COLS)
print()
print('Split definition:')
print(f"  TRAIN      : year <= 2021   (n={int(train_idx.sum())})")
print(f"  VALIDATION : year == 2022   (n={int(val_idx.sum())})")
print(f"  ADAPT/TEST : year == 2023   (n={int(test23_idx.sum())})")
print(f"  FINAL TEST : year == 2024   (n={int(final24_idx.sum())}, touched only in Section 26)")
print()
print('Best model selected on VAL(2022) only:', best_model_name)


**Untuk menjalankan ulang:** letakkan `eurepoc_global_dataset_1_3.csv` di `RAW_PATH`
di atas dan jalankan semua sel dari atas ke bawah; setiap hasil di notebook ini dihasilkan
ulang dari satu file tersebut tanpa state eksternal, tanpa file perantara yang di-cache, dan
tanpa angka yang diketik manual.


## 30. Reproducibility

### Konfigurasi yang di-assert

Nilai berikut diperiksa saat runtime supaya narasi di Discussion tidak pernah
melaporkan konfigurasi yang berbeda dari yang benar-benar dijalankan:

Assert dijalankan di Bagian 16 (Temporal Holdout, `EXPECTED_N_MODELS`) untuk jumlah
model dan di Bagian 22 (Explainability, `EXPECTED_IMPORTANCE_METHOD`) untuk metode
importance. `EXPECTED_*` diisi dari hasil
eksekusi pertama dan diperbarui hanya bila konfigurasi paket memang berubah. Di
lingkungan tanpa `xgboost`/`lightgbm`/`shap`, nilainya adalah `4` dan
`'permutation importance (Macro F1 drop)'`.

### Artefak yang disimpan

| berkas | isi |
|---|---|
| `out/missingness_by_period.csv` | matriks missingness per periode (S1.2) |
| `out/category_vocab.json` | vocabulary kategori multi-hot, di-fit dari TRAIN |
| `out/top_k_regions.json` | 12 tag wilayah, dipilih dari TRAIN |

### Cara menjalankan ulang

```bash
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace \
    EuRepoC_Temporal_Robustness_Research_SPRINT.ipynb
```

Berkas data dicari berurutan di: `$EUREPOC_WORK_DIR`, `./dataset/`, lalu path notebook.
Tidak ada path absolut ke satu mesin yang tertanam di kode.

### Random seed

`RANDOM_SEED = 42` dipakai untuk seluruh estimator, split, dan bootstrap. Analisis
variabilitas antar-seed (S5.3) menjalankan 5 seed tambahan dan melaporkan rentangnya,
bukan satu angka tunggal.
